# Datathon Fase 5 — Associação Passos Mágicos
### Risco de entrada em defasagem escolar: dados, método, resultados oficiais e limites

Notebook final e **reproduzível** do projeto. Ele **apresenta** o trabalho já concluído; não treina, não
reavalia e não recalibra nada. Todos os números são lidos dos artefatos oficiais versionados
(`artifacts/`, `reports/`, `docs/`) ou das funções e constantes de `src/`.

## 1. Título e resumo executivo

**Como executar (Windows e Linux/macOS)**

1. Python 3.12 e as dependências fixadas em `requirements.txt` (mesmas versões do congelamento do modelo).
2. Dependências extras para *executar notebooks*: `notebooks/requirements-notebook.txt`.
3. Abrir este arquivo a partir de qualquer pasta dentro do repositório (VS Code, JupyterLab ou
   `python -m jupyter nbconvert --to notebook --execute notebooks/datathon_fase5.ipynb`).
   A raiz do projeto é localizada automaticamente; nenhum caminho absoluto é usado.

```text
python -m venv .venv
.venv/Scripts/python -m pip install -r requirements.txt -r notebooks/requirements-notebook.txt   # Windows
.venv/bin/python        -m pip install -r requirements.txt -r notebooks/requirements-notebook.txt   # Linux/macOS
```

**O que depende de dados privados.** Somente a *validação completa opcional* (seção 1, "Integridade")
reconfere hashes de `DATATHON/` e `local_data/`, que **não** são distribuídos no Git. Sem eles, o notebook
executa por inteiro com os artefatos públicos e informa que essa etapa foi dispensada.

**Convenções.** Números seguem o formato dos relatórios oficiais (ponto decimal). Nenhuma tabela mostra
linhas individuais: apenas agregados, com as supressões de grupos pequenos já aplicadas nos relatórios.

In [1]:
# ---- Preparação do ambiente: raiz do repositório, módulos de src/ e artefatos (somente leitura) ----
import sys
from importlib import metadata
from pathlib import Path

from IPython.display import Markdown, display

sys.dont_write_bytecode = True  # não gravar caches dentro do repositório


def localizar_raiz(inicio=None):
    inicio = Path(inicio or Path.cwd()).resolve()
    for pasta in (inicio, *inicio.parents):
        if (pasta / "src" / "modelagem.py").is_file() and (pasta / "docs" / "TASKS.md").is_file():
            return pasta
    raise FileNotFoundError(
        "Raiz do repositório não encontrada. Abra o notebook dentro do projeto fiap-datathon-fase5 "
        "(a pasta que contém src/, docs/ e artifacts/).")


ROOT = localizar_raiz()
sys.path.insert(0, str(ROOT / "src"))

try:
    import modelagem
    import preparacao_coortes as coortes
    import analises_negocio as analises
    import rastreabilidade as rastro
except ImportError as erro:
    raise RuntimeError(
        f"Dependência ausente ({erro.name}). Instale: pip install -r requirements.txt "
        "-r notebooks/requirements-notebook.txt") from erro

PUBLICOS = [
    "artifacts/configuracao_congelada.json", "artifacts/schema_modelo.json",
    "artifacts/avaliacao_temporal.json", "artifacts/modelo_avaliado.joblib", "artifacts_meta.json",
    "reports/metadados_preparacao.json", "reports/metadados_coortes.json",
    "reports/metricas_modelagem.json", "reports/metricas_analises_negocio.json",
    "reports/relatorio_analises_negocio.md", "reports/relatorio_modelagem.md",
    "reports/relatorio_coortes_modelagem.md", "reports/relatorio_preparacao_inicial.md",
    "reports/relatorio_auditoria_inicial.md", "reports/curvas_modelagem.png",
    "docs/contrato_metodologico.md", "docs/TASKS.md",
]
ausentes = [caminho for caminho in PUBLICOS if not (ROOT / caminho).is_file()]
if ausentes:
    raise FileNotFoundError(
        "Artefatos públicos ausentes: " + ", ".join(ausentes)
        + ". Use o clone completo da branch do projeto; este notebook não os regenera.")


# ---- auxiliares de apresentação (não contêm lógica de dados) ----
def ler(caminho):
    return modelagem.read_json(ROOT / caminho)


def mostrar(texto):
    display(Markdown(texto))


def num(valor, casas=4):
    return "não estimável" if valor is None else f"{valor:.{casas}f}"


def pct(valor, casas=1):
    return "—" if valor is None else f"{100 * valor:.{casas}f}%"


def tabela(cabecalhos, linhas):
    saida = ["| " + " | ".join(cabecalhos) + " |", "| " + " | ".join("---" for _ in cabecalhos) + " |"]
    saida += ["| " + " | ".join(str(c) for c in linha) + " |" for linha in linhas]
    return "\n".join(saida)


# ---- artefatos oficiais, carregados uma única vez ----
CONGELADO = ler(modelagem.FREEZE)
SCHEMA = ler(modelagem.SCHEMA)
ACESSO = ler(modelagem.ACCESS)
MET = ler(modelagem.METRICS)
AUDIT = ler("artifacts_meta.json")
PREP = ler("reports/metadados_preparacao.json")
COORTES = ler(coortes.META)
NEGOCIO = ler(analises.METRICS)
MODELO = analises.official_model(ROOT)  # já confere limiar e hash de configuração entre os artefatos
RESUMO = PREP["summary"]
ANOS = ("2022", "2023", "2024")
DEV, TESTE = MET["oof"]["metricas"], MET["temporal"]["metricas"]
COR = COORTES["summary"]["coortes"]

mostrar(f"Raiz do projeto localizada (relativa): `.` — {len(PUBLICOS)} artefatos públicos encontrados. "
        f"Sistema: {sys.platform}; Python {sys.version.split()[0]}.")

Raiz do projeto localizada (relativa): `.` — 17 artefatos públicos encontrados. Sistema: win32; Python 3.12.9.

### Ambiente e integridade (validação antes de qualquer análise)

**O que foi feito.** Conferimos (a) se as versões dos pacotes são as do congelamento do modelo,
(b) se os artefatos oficiais coincidem com os hashes SHA-256 registrados e (c) a ordem cronológica
*congelamento → abertura única do teste → conclusão*. **Por quê.** Garante que o notebook mostra os
resultados oficiais e não uma reexecução. **Limitação.** As versões divergentes geram aviso, não bloqueio:
o notebook só *lê* JSON e imagens; contudo, versões diferentes impedem a verificação formal do congelamento.

In [2]:
pacotes_congelados = CONGELADO["configuracao"]["packages"]
linhas = []
for pacote, esperada in pacotes_congelados.items():
    try:
        atual = metadata.version(pacote)
    except metadata.PackageNotFoundError:
        atual = "não instalado"
    linhas.append([pacote, esperada, atual, "ok" if atual == esperada else "DIVERGE"])
mostrar(tabela(["Pacote", "Versão congelada", "Versão neste ambiente", "Situação"], linhas))
VERSOES_OK = all(l[3] == "ok" for l in linhas)
if not VERSOES_OK:
    mostrar("> **Aviso:** versões diferentes das congeladas. Use `requirements.txt` para a reprodução formal.")

verificacoes = []


def verificar(nome, funcao):
    try:
        funcao()
        verificacoes.append([nome, "aprovada"])
    except Exception as erro:  # mensagem curta e sem caminhos
        verificacoes.append([nome, f"FALHOU: {type(erro).__name__}"])


verificar("Configuração congelada, schema, modelo avaliado e contrato (hashes)",
          lambda: modelagem.validate_frozen(ROOT))
verificar("Saídas registradas na abertura única do teste (hashes)",
          lambda: modelagem.verify_hashes(ROOT, ACESSO["output_hashes"]))
verificar("Relatório e dez figuras das análises (hashes)",
          lambda: modelagem.verify_hashes(ROOT, NEGOCIO["output_hashes"]))
verificar("Métricas oficiais coerentes entre relatório, schema e congelamento",
          lambda: analises.official_model(ROOT))
def conteudo_publico_portatil():
    for arquivo in (analises.REPORT, modelagem.REPORT):
        if rastro.public_text_issues((ROOT / arquivo).read_text(encoding="utf-8")):
            raise ValueError("conteúdo não portátil")


verificar("Conteúdo público sem caminhos locais ou identificadores", conteudo_publico_portatil)
cronologia_ok = CONGELADO["congelado_em"] < ACESSO["aberta_em"] < ACESSO["concluida_em"] and ACESSO["status"] == "concluida"
verificacoes.append(["Cronologia: congelamento < abertura do teste < conclusão", "aprovada" if cronologia_ok else "FALHOU"])
mostrar(tabela(["Verificação (somente leitura)", "Resultado"], verificacoes))

PRIVADO = (ROOT / "DATATHON").is_dir() and (ROOT / "local_data" / "base_longitudinal.jsonl").is_file()
if PRIVADO:
    completa = []
    for nome, funcao in (("Modelagem: entradas privadas, OOF e avaliação", modelagem.validate_artifacts),
                         ("Análises: fontes, entradas e saídas", analises.validate_artifacts)):
        try:
            funcao(ROOT)
            completa.append([nome, "aprovada"])
        except Exception as erro:
            completa.append([nome, f"FALHOU: {type(erro).__name__}"])
    mostrar("**Validação completa opcional** (usa os validadores de `src/`, somente leitura; "
            "exibe apenas aprovado/falhou):\n\n" + tabela(["Validador", "Resultado"], completa))
else:
    mostrar("> **Fonte privada não disponível neste ambiente** (`DATATHON/` e/ou `local_data/`). "
            "A validação completa opcional foi dispensada; todas as seções abaixo usam apenas artefatos públicos. "
            "Para executá-la, disponibilize os originais localmente conforme o `README.md`.")

| Pacote | Versão congelada | Versão neste ambiente | Situação |
| --- | --- | --- | --- |
| numpy | 2.5.3 | 2.5.3 | ok |
| pandas | 3.0.5 | 3.0.5 | ok |
| scikit-learn | 1.9.1 | 1.9.1 | ok |
| scipy | 1.18.1 | 1.18.1 | ok |
| joblib | 1.6.0 | 1.6.0 | ok |
| matplotlib | 3.11.2 | 3.11.2 | ok |

| Verificação (somente leitura) | Resultado |
| --- | --- |
| Configuração congelada, schema, modelo avaliado e contrato (hashes) | aprovada |
| Saídas registradas na abertura única do teste (hashes) | aprovada |
| Relatório e dez figuras das análises (hashes) | aprovada |
| Métricas oficiais coerentes entre relatório, schema e congelamento | aprovada |
| Conteúdo público sem caminhos locais ou identificadores | aprovada |
| Cronologia: congelamento < abertura do teste < conclusão | aprovada |

**Validação completa opcional** (usa os validadores de `src/`, somente leitura; exibe apenas aprovado/falhou):

| Validador | Resultado |
| --- | --- |
| Modelagem: entradas privadas, OOF e avaliação | aprovada |
| Análises: fontes, entradas e saídas | aprovada |

### Resumo executivo

In [3]:
ic = MET["robustez"]["intervalos"]["completo"]
mostrar(f'''
- **Base:** {RESUMO["total_registros"]} registros anuais (PEDE 2022, 2023 e 2024) de {AUDIT["inventory_count"]} fontes auditadas;
  {RESUMO["ra_nos_tres_anos"]} identificadores aparecem nos três anos. Totais anuais **não** são alunos únicos.
- **Pergunta preditiva:** entre alunos *sem defasagem* no ano *t*, quem **entra em defasagem** no ano *t+1*?
- **Dados de modelagem:** desenvolvimento 2022→2023 ({DEV["n"]} transições, {DEV["eventos"]} eventos) e
  teste temporal 2023→2024 ({TESTE["n"]} transições, {TESTE["eventos"]} eventos).
- **Modelo oficialmente avaliado:** {MODELO["modelo"]} (regressão logística regularizada), sete preditores, limiar congelado {MODELO["limiar"]:.4f}.
- **Resultado no teste temporal único:** AP {TESTE["average_precision"]:.3f}, ROC-AUC {TESTE["roc_auc"]:.3f},
  precisão {TESTE["precisao"]:.3f}, recall {TESTE["recall"]:.3f} (IC95% {ic["recall"]["ic95"][0]:.3f} a {ic["recall"]["ic95"][1]:.3f}).
- **Principal alerta:** o recall foi {pct(DEV["recall"])} no desenvolvimento (validação interna) e {pct(TESTE["recall"])} no teste —
  a meta interna de sensibilidade **não se manteve** no ano seguinte.
- **Análises de negócio:** {len(NEGOCIO["perguntas"])} perguntas respondidas e {len(NEGOCIO["figuras"])} gráficos agregados; todas as relações são
  **associações observacionais**, sem afirmação de causa.
- **Uso adequado:** apoio à priorização e à revisão humana; não é diagnóstico nem decisão automática.
''')


- **Base:** 3030 registros anuais (PEDE 2022, 2023 e 2024) de 13 fontes auditadas;
  468 identificadores aparecem nos três anos. Totais anuais **não** são alunos únicos.
- **Pergunta preditiva:** entre alunos *sem defasagem* no ano *t*, quem **entra em defasagem** no ano *t+1*?
- **Dados de modelagem:** desenvolvimento 2022→2023 (189 transições, 60 eventos) e
  teste temporal 2023→2024 (311 transições, 84 eventos).
- **Modelo oficialmente avaliado:** logistica (regressão logística regularizada), sete preditores, limiar congelado 0.2670.
- **Resultado no teste temporal único:** AP 0.621, ROC-AUC 0.804,
  precisão 0.642, recall 0.405 (IC95% 0.302 a 0.512).
- **Principal alerta:** o recall foi 81.7% no desenvolvimento (validação interna) e 40.5% no teste —
  a meta interna de sensibilidade **não se manteve** no ano seguinte.
- **Análises de negócio:** 11 perguntas respondidas e 10 gráficos agregados; todas as relações são
  **associações observacionais**, sem afirmação de causa.
- **Uso adequado:** apoio à priorização e à revisão humana; não é diagnóstico nem decisão automática.


## 2. Contexto da Associação Passos Mágicos

**O que foi feito.** Reunimos, sem acrescentar informação externa, o que o material do desafio informa.
**Por quê.** Situar quem lê antes dos números. **Resultado.** O enunciado do Datathon apresenta o caso
*Passos Mágicos* e pede a análise dos anos de 2022, 2023 e 2024. O material institucional fornecido
(`desvendando_passos.pdf`) descreve o percurso de admissão e acompanhamento — inscrição, prova de sondagem,
entrevistas, consenso psicopedagógico sobre o nível de adequação, avaliação socioeconômica e matrícula —
e o dicionário/relatórios PEDE descrevem os indicadores usados neste trabalho:

| Indicador | Leitura (conforme a documentação do projeto) |
| --- | --- |
| **IAN** | adequação ao nível; derivado diretamente da defasagem registrada |
| **IDA** | desempenho acadêmico |
| **IEG** | engajamento |
| **IAA** | autoavaliação |
| **IPS** | aspecto psicossocial |
| **IPP** | aspecto psicopedagógico (ausente por construção em 2022) |
| **IPV** | ponto de virada |
| **INDE** | índice composto ponderado dos indicadores, com regras por fase |

*Defasagem* (D) é a diferença registrada entre a fase efetiva e a ideal; `D < 0` indica defasagem.
**Limitações.** O significado da fase 9 e de valores "INCLUIR" não está documentado; a equivalência
curricular entre anos não foi confirmada. O trabalho não conclui sobre o programa em si: descreve
associações e testa uma previsão. Detalhes: `docs/evidencias_documentais.md`.

## 3. Objetivo do Datathon

**O que foi feito.** Traduzimos o desafio em três entregas analíticas. **Por quê.** Para que cada seção do
notebook responda a um pedido explícito do enunciado. **Resultado.**

1. **Análise dos indicadores** (11 perguntas de negócio, seções 17 e 18): defasagem, desempenho, engajamento,
   autoavaliação, aspectos psicossociais e psicopedagógicos, ponto de virada, INDE, Pedras e insights.
2. **Modelo de probabilidade de risco de defasagem** (seções 8 a 16), com desenho temporal e avaliação única.
3. **Comunicação e reprodutibilidade** (este notebook); aplicação, apresentação e vídeo são etapas posteriores
   do plano (`docs/TASKS.md`).

**Limitações.** O enunciado adverte que análises observacionais não devem presumir causalidade; este trabalho segue essa regra.

## 4. Fontes e proteção dos dados

**O que foi feito.** Todas as fontes originais ficam em `DATATHON/`, tratadas como **somente leitura**; bases
derivadas individuais ficam em `local_data/`, cópias de segurança em `local_recovery/`. Nenhum desses
diretórios é versionado, e os scripts verificam isso antes de gravar. **Por quê.** Os dados descrevem crianças e
adolescentes; identificadores (RA, nome) nunca são preditores nem aparecem em documentos públicos.
**Resultado.**

In [4]:
priv = AUDIT["privacy"]
mostrar(tabela(["Verificação de proteção (metadados da auditoria)", "Resultado"], [
    ["Diretórios excluídos do versionamento", ", ".join(f"`{Path(d).parts[0]}/`" for d in priv["diretorios_ignorados"])],
    ["Arquivos individuais rastreados pelo Git", priv["arquivos_individuais_rastreados"]],
    ["Dados individuais fora do versionamento", priv["dados_individuais_fora_versionamento"]],
    ["Fontes idênticas antes e depois da execução da auditoria", AUDIT["source_integrity_preserved"]],
    ["Documentos manuais preservados", AUDIT["manual_documents_preserved"]],
    ["Parâmetros de supressão nas análises (mínimo de observações por perfil)", NEGOCIO["privacidade"]["minimo_perfil"]],
]))

| Verificação de proteção (metadados da auditoria) | Resultado |
| --- | --- |
| Diretórios excluídos do versionamento | `DATATHON/`, `local_data/`, `local_recovery/` |
| Arquivos individuais rastreados pelo Git | 0 |
| Dados individuais fora do versionamento | True |
| Fontes idênticas antes e depois da execução da auditoria | True |
| Documentos manuais preservados | True |
| Parâmetros de supressão nas análises (mínimo de observações por perfil) | 10 |

**Limitações.** A proteção é por agregação e supressão de células pequenas; ela reduz, mas não elimina, o
risco de reidentificação por combinação de informações externas. Por isso os resultados devem circular
apenas em formato agregado.

## 5. Auditoria das 13 fontes

**O que foi feito.** Inventário de todos os arquivos recebidos, com hash SHA-256 (comparação binária exata)
antes e depois da execução, e auditoria linha a linha da planilha principal. **Por quê.** Provar que os
originais não foram alterados e que cada número decorre de uma fonte identificável. Somente a **planilha
principal** alimenta as análises; os demais arquivos (relatórios anteriores, bases antigas, dicionário)
são material de contexto e **não foram combinados à base**. **Resultado.**

In [5]:
fontes = AUDIT["source_hashes_after"]
identicas = AUDIT["source_hashes_before"] == fontes
linhas = [[i, nome, Path(nome).suffix.lstrip(".").upper(), digest[:16] + "…"]
          for i, (nome, digest) in enumerate(fontes.items(), 1)]
mostrar(f"**{len(fontes)} fontes inventariadas** (esperado: {AUDIT['inventory_count']}); "
        f"hashes antes = depois: **{identicas}**; "
        f"diferença em relação ao inventário anterior: {AUDIT['inventory_diff_from_previous_run']}.")
mostrar(tabela(["#", "Arquivo (nome relativo)", "Tipo", "SHA-256 (16 primeiros caracteres)"], linhas))

linhas = []
for ano in ANOS:
    a = RESUMO["anos"][ano]
    linhas.append([ano, a["registros"], a["colunas"], a["defasagem"]["D>=0"], a["defasagem"]["D<0"],
                   a["ra_status"].get("valido", 0), a["registros_ra_ano_duplicado"]])
mostrar("**Resultado da auditoria da planilha principal, por ano:**\n\n"
        + tabela(["Ano", "Registros", "Colunas", "Sem defasagem (D≥0)", "Com defasagem (D<0)", "RA válidos", "RA duplicados no ano"], linhas))

ok = [r for r in RESUMO["referencias"] if r["confere"]]
mostrar(f"Conferência com valores de referência do enunciado/dados: **{len(ok)} de {len(RESUMO['referencias'])}** reproduzidas por código.")

**13 fontes inventariadas** (esperado: 13); hashes antes = depois: **True**; diferença em relação ao inventário anterior: {'adicionados': [], 'removidos': [], 'alterados': []}.

| # | Arquivo (nome relativo) | Tipo | SHA-256 (16 primeiros caracteres) |
| --- | --- | --- | --- |
| 1 | BASE DE DADOS PEDE 2024 - DATATHON.xlsx | XLSX | 05779754185f7c9c… |
| 2 | Bases antigas/Base de dados - Passos Mágicos.zip | ZIP | 71cc75f1fbd4b4f1… |
| 3 | Bases antigas/DATATHON-PASSOS-MÁGICOS.ipynb | IPYNB | 47b93c9d47b5686b… |
| 4 | Bases antigas/PEDE_PASSOS_DATASET_FIAP.csv | CSV | 28f6563b2feb849c… |
| 5 | Bases antigas/PEDE_PASSOS_DATASET_FIAP.xlsx | XLSX | 67509da73f74d1e0… |
| 6 | desvendando_passos.pdf | PDF | 594694ae2859bd92… |
| 7 | Dicionário Dados Datathon.pdf | PDF | c141a779a02d34e8… |
| 8 | Links adicionais da passos.docx | DOCX | 5d72421f1d3f3e82… |
| 9 | PEDE_ Pontos importantes.docx | DOCX | ee2a6cd6e0d53f91… |
| 10 | POSTECH - Datathon - Fase 5 (1).pdf | PDF | ac333680d88bc7d3… |
| 11 | Relatorio PEDE2022.pdf | PDF | 34314dcc09fe3686… |
| 12 | Relatório PEDE2020.pdf | PDF | 618599418db1b9f8… |
| 13 | Relatório PEDE2021.pdf | PDF | 859ce95807a5508d… |

**Resultado da auditoria da planilha principal, por ano:**

| Ano | Registros | Colunas | Sem defasagem (D≥0) | Com defasagem (D<0) | RA válidos | RA duplicados no ano |
| --- | --- | --- | --- | --- | --- | --- |
| 2022 | 860 | 42 | 259 | 601 | 860 | 0 |
| 2023 | 1014 | 48 | 462 | 552 | 1014 | 0 |
| 2024 | 1156 | 50 | 622 | 534 | 1156 | 0 |

Conferência com valores de referência do enunciado/dados: **28 de 28** reproduzidas por código.

**Limitações.** Erros de planilha (`#N/A`, `#DIV/0!`), células vazias e textos foram **preservados como
indisponíveis**, sem converter em zero. Datas de nascimento e idades têm formatos mistos entre anos e não
são usadas no modelo. Equivalência curricular entre anos, fase 9 e "INCLUIR" seguem pendentes de definição
documental (`docs/revisao_auditoria.md`).

## 6. Preparação longitudinal de 2022, 2023 e 2024

**O que foi feito.** Consolidação das três abas anuais em **um registro por aluno-ano**, com tipos lidos
diretamente da planilha, padronização mínima de campos e ligação entre anos **somente por RA válido e único**
(junção `one_to_one`; nunca por nome ou posição). Fase alfanumérica extraída de forma explícita (ALFA = 0).
**Por quê.** Permite comparar o mesmo aluno entre anos sem inventar dados: ausências permanecem ausências.
**Resultado.**

In [6]:
v = PREP["validations"]
mostrar(tabela(["Validação da preparação", "Resultado"], [
    ["Registros conferidos contra a origem", v["registros_conferidos_na_origem"]],
    ["Células originais conferidas", v["celulas_originais_conferidas"]],
    ["Correspondência integral com a origem", v["correspondencia_integral_origem"]],
    ["RA único por ano; registros com RA inválido", f'{v["ra_ano_unico"]}; {v["registros_com_ra_invalido"]}'],
    ["Arquivos derivados reabertos e idênticos", f'{v["jsonl_reaberto_identico"]} / {v["csv_reaberto_campos_conferidos"]}'],
    ["Referências reproduzidas", f'{v["referencias_conferidas"]} de {v["referencias_total"]}'],
    ["Fluxos de transição fecham", v["fluxos_transicoes_fecham"]],
]))

linhas = []
for chave, t in RESUMO["transicoes"].items():
    r = t["fases_0_a_7"]
    linhas.append([chave.replace("->", "→"), r["eligible_count"], r["with_dest_count"],
                   r["not_found_in_destination_count"], r["dest_defasado_count"]])
mostrar("**Transições por RA (alunos sem defasagem na origem, fases 0 a 7):**\n\n"
        + tabela(["Transição", "Elegíveis na origem", "Encontrados no ano seguinte", "Não encontrados", "Com defasagem no destino"], linhas))

| Validação da preparação | Resultado |
| --- | --- |
| Registros conferidos contra a origem | 3030 |
| Células originais conferidas | 142592 |
| Correspondência integral com a origem | True |
| RA único por ano; registros com RA inválido | True; 0 |
| Arquivos derivados reabertos e idênticos | True / True |
| Referências reproduzidas | 28 de 28 |
| Fluxos de transição fecham | True |

**Transições por RA (alunos sem defasagem na origem, fases 0 a 7):**

| Transição | Elegíveis na origem | Encontrados no ano seguinte | Não encontrados | Com defasagem no destino |
| --- | --- | --- | --- | --- |
| 2022→2023 | 259 | 189 | 70 | 60 |
| 2023→2024 | 399 | 311 | 88 | 84 |

**Limitações.** Alunos que saem da base entre anos ficam sem desfecho (não são tratados como "sem
defasagem"). Indicadores compostos (INDE, Pedra) não foram recalculados. O IPP de 2022 é ausência
**estrutural** e nunca foi convertido em zero.

## 7. Características dos 3.030 registros

**O que foi feito.** Descrição agregada da base longitudinal. **Por quê.** Entender cobertura e qualidade
antes de interpretar qualquer resultado. **Resultado.** (Cada célula mostra *disponíveis / total*.)

In [7]:
total = RESUMO["total_registros"]
por_ano = {a: RESUMO["anos"][a]["registros"] for a in ANOS}
comuns = {r["verificacao"]: r["calculado"] for r in RESUMO["referencias"]}
mostrar(f"**{total} registros anuais** = " + " + ".join(f"{por_ano[a]} ({a})" for a in ANOS)
        + f". Identificadores presentes nos três anos: **{RESUMO['ra_nos_tres_anos']}**; "
        f"comuns 2022→2023: **{comuns['RA comuns 2022->2023']}**; comuns 2023→2024: **{comuns['RA comuns 2023->2024']}**. "
        "Como um aluno pode aparecer em vários anos, estes totais **não** contam alunos únicos.")

indicadores = ["IAN", "IDA", "IEG", "IAA", "IPS", "IPP", "IPV", "INDE"]
tab = {(i["ano"], i["indicador"]): i for i in RESUMO["indicadores"]}
linhas = []
for ind in indicadores:
    linha = [ind]
    for ano in ANOS:
        item = tab[(int(ano), ind)]
        linha.append(f'{item["disponiveis"]} / {item["total"]}')
    motivos = sorted({m.split(":")[0].replace("erro_excel", "erro de planilha") for ano in ANOS
                      for m in tab[(int(ano), ind)]["motivos"]})
    linha.append(", ".join(motivos) if motivos else "—")
    linhas.append(linha)
mostrar("**Disponibilidade dos oito indicadores (valores numéricos válidos):**\n\n"
        + tabela(["Indicador", "2022", "2023", "2024", "Tipos de indisponibilidade observados"], linhas))

linhas = []
for ano in ("2022", "2023"):
    fases = RESUMO["anos"][ano]["fases_extraidas"]
    linhas.append([ano, ", ".join(f"fase {k}: {n}" for k, n in fases.items())])
mostrar("**Distribuição por fase de origem (anos usados como origem nas coortes):**\n\n"
        + tabela(["Ano", "Registros por fase"], linhas))

**3030 registros anuais** = 860 (2022) + 1014 (2023) + 1156 (2024). Identificadores presentes nos três anos: **468**; comuns 2022→2023: **600**; comuns 2023→2024: **765**. Como um aluno pode aparecer em vários anos, estes totais **não** contam alunos únicos.

**Disponibilidade dos oito indicadores (valores numéricos válidos):**

| Indicador | 2022 | 2023 | 2024 | Tipos de indisponibilidade observados |
| --- | --- | --- | --- | --- |
| IAN | 860 / 860 | 1014 / 1014 | 1156 / 1156 | — |
| IDA | 860 / 860 | 937 / 1014 | 1055 / 1156 | celula_vazia, erro de planilha |
| IEG | 860 / 860 | 938 / 1014 | 1156 / 1156 | celula_vazia |
| IAA | 860 / 860 | 951 / 1014 | 1054 / 1156 | celula_vazia, texto_vazio |
| IPS | 860 / 860 | 945 / 1014 | 1054 / 1156 | erro de planilha |
| IPP | 0 / 860 | 938 / 1014 | 1054 / 1156 | ausencia_estrutural, erro de planilha |
| IPV | 860 / 860 | 938 / 1014 | 1054 / 1156 | erro de planilha |
| INDE | 860 / 860 | 931 / 1014 | 1054 / 1156 | erro de planilha, incluir |

**Distribuição por fase de origem (anos usados como origem nas coortes):**

| Ano | Registros por fase |
| --- | --- |
| 2022 | fase 0: 190, fase 1: 192, fase 2: 155, fase 3: 148, fase 4: 76, fase 5: 60, fase 6: 18, fase 7: 21 |
| 2023 | fase 0: 231, fase 1: 173, fase 2: 200, fase 3: 132, fase 4: 94, fase 5: 65, fase 6: 33, fase 7: 23, fase 8: 63 |

**Limitações.** A maior parte da indisponibilidade está em 2023 e 2024 (erros de planilha e células vazias),
e o IPP não existe em 2022. Extremos fora da faixa operacional 0–10 foram **sinalizados, não corrigidos**.
Fases 8 e 9 têm regras/significado próprios e ficam fora do modelo. Comparações entre anos misturam mudanças
de composição e mudanças dos alunos.

## 8. Contrato metodológico

**O que foi feito.** Antes de treinar qualquer modelo, o grupo aprovou (17/09/2026) um contrato escrito
(`docs/contrato_metodologico.md`) com problema, população, alvo, preditores, tratamento de ausências,
modelos, métricas, limiar e regras de incerteza. **Por quê.** Fixar as regras *antes* de ver o teste
evita escolhas guiadas pelo resultado. **Resultado.** Síntese das regras:

| Tema | Regra congelada |
| --- | --- |
| Unidade de análise | transição aluno-ano (RA usado só para ligar anos; nunca é preditor) |
| Divisão temporal | desenvolvimento 2022→2023; teste final 2023→2024, sem embaralhar anos |
| Seleção de modelo | somente no desenvolvimento, validação cruzada estratificada (5 dobras, semente 42) |
| Ausências | imputação pela mediana **dentro** de cada dobra; nada convertido em zero |
| Métrica principal | *Average Precision* (AP); também ROC-AUC, precisão, recall, F1, Brier e calibração |
| Limiar | escolhido no desenvolvimento, recall mínimo de 80%, maior precisão; **não** muda no teste |
| Incerteza | *bootstrap* percentil no teste (2.000 reamostragens, semente 42, 95%) |
| Uso | preditivo e observacional; sem causalidade; apoio à decisão humana |

In [8]:
contrato = ROOT / "docs" / "contrato_metodologico.md"
registrado = CONGELADO["configuracao"]["contrato_sha256"]
mostrar(f"Contrato conferido contra o hash registrado no congelamento (tolerando apenas LF/CRLF): "
        f"**{rastro.file_matches_sha256(contrato, registrado)}**. "
        f"Protocolo do congelamento igual ao protocolo implementado em `src/modelagem.py`: "
        f"**{CONGELADO['configuracao']['protocolo'] == modelagem.PROTOCOL}**.")

Contrato conferido contra o hash registrado no congelamento (tolerando apenas LF/CRLF): **True**. Protocolo do congelamento igual ao protocolo implementado em `src/modelagem.py`: **True**.

**Limitações.** O contrato prevê um **modelo operacional** futuro retreinado com as duas transições; ele **não
foi criado** e, se for, não herdará as métricas oficiais deste modelo avaliado.

## 9. Definição da variável-alvo

**O que foi feito.** O alvo mede a **entrada em defasagem**: para um aluno *sem defasagem* no ano *t*
(`D ≥ 0`), `y = 1` se no ano *t+1* a defasagem registrada for negativa (`D < 0`) e `y = 0` caso contrário.
Sem observação futura válida, o alvo é **desconhecido** e o aluno não entra no treino nem na avaliação.
**Por quê.** É um desfecho observável, definido a partir de um campo já registrado, que orienta acompanhamento
preventivo. **Resultado.**

In [9]:
regra = COORTES["cohort_schema"]["y"]
mostrar(tabela(["Situação no ano de destino", "Alvo"], [
    ["Defasagem registrada negativa (" + regra["positive"].replace("D_destino", "D no destino") + ")", "1"],
    ["Defasagem registrada ≥ 0 (" + regra["negative"].replace("D_destino", "D no destino") + ")", "0"],
    ["Aluno ausente no destino ou defasagem indisponível", "desconhecido (fora do modelo)"],
]))
linhas = []
for nome, rotulo in (("desenvolvimento", "Desenvolvimento 2022→2023"), ("teste_temporal", "Teste temporal 2023→2024")):
    c = COR[nome]
    linhas.append([rotulo, c["supervisionados"], c["alvo_1"], c["alvo_0"], c["alvo_desconhecido"], pct(c["taxa_evento"], 2)])
mostrar(tabela(["Coorte", "Supervisionados", "Alvo = 1", "Alvo = 0", "Alvo desconhecido (excluído)", "Taxa de evento"], linhas))

| Situação no ano de destino | Alvo |
| --- | --- |
| Defasagem registrada negativa (D no destino < 0) | 1 |
| Defasagem registrada ≥ 0 (D no destino >= 0) | 0 |
| Aluno ausente no destino ou defasagem indisponível | desconhecido (fora do modelo) |

| Coorte | Supervisionados | Alvo = 1 | Alvo = 0 | Alvo desconhecido (excluído) | Taxa de evento |
| --- | --- | --- | --- | --- | --- |
| Desenvolvimento 2022→2023 | 189 | 60 | 129 | 70 | 31.75% |
| Teste temporal 2023→2024 | 311 | 84 | 227 | 88 | 27.01% |

**Limitações.** O alvo é *entrada em defasagem entre elegíveis*, não qualquer queda de desempenho. Quem
já estava defasado na origem fica fora deste modelo (permanece nas análises descritivas). "Sem defasagem no
destino" pode refletir mudança de fase ou de registro; e o desaparecimento da base **não** é interpretado
como sucesso nem fracasso.

## 10. Apresentação dos sete preditores aprovados

**O que foi feito.** Lista **fechada** de sete variáveis, todas conhecidas no ano de origem *t*. **Por quê.**
Cobrem desempenho, engajamento, autoavaliação, aspecto psicossocial, ponto de virada, fase e defasagem
já registrada, sem depender do futuro. **Resultado.**

In [10]:
descricao = {"ida": "desempenho acadêmico (IDA)", "ieg": "engajamento (IEG)", "iaa": "autoavaliação (IAA)",
             "ips": "aspecto psicossocial (IPS)", "ipv": "ponto de virada (IPV)",
             "fase_origem": "fase educacional no ano de origem (categórica, 0 a 7)",
             "defasagem_origem": "defasagem registrada no ano de origem"}
tipos = SCHEMA["tipos"]
mostrar(tabela(["#", "Preditor", "Descrição", "Tipo no schema"],
               [[i, f"`{c}`", descricao[c], tipos[c]] for i, c in enumerate(SCHEMA["colunas"], 1)]))
mostrar(f"Ordem obrigatória do schema igual à constante `FEATURES` de `src/preparacao_coortes.py`: "
        f"**{SCHEMA['colunas'] == list(coortes.FEATURES)}**. Quantidade de preditores: **{len(SCHEMA['colunas'])}**.")
mostrar("**Explicitamente fora do modelo:** " + ", ".join(COORTES["criteria"]["excluded"])
        + ". (Ex.: IAN é derivado da defasagem; INDE e Pedra compõem-se dos próprios indicadores; gênero só serviria a uma auditoria de equidade e não está nas coortes.)")

| # | Preditor | Descrição | Tipo no schema |
| --- | --- | --- | --- |
| 1 | `ida` | desempenho acadêmico (IDA) | numero anulavel |
| 2 | `ieg` | engajamento (IEG) | numero anulavel |
| 3 | `iaa` | autoavaliação (IAA) | numero anulavel |
| 4 | `ips` | aspecto psicossocial (IPS) | numero anulavel |
| 5 | `ipv` | ponto de virada (IPV) | numero anulavel |
| 6 | `fase_origem` | fase educacional no ano de origem (categórica, 0 a 7) | string categorica |
| 7 | `defasagem_origem` | defasagem registrada no ano de origem | numero anulavel |

Ordem obrigatória do schema igual à constante `FEATURES` de `src/preparacao_coortes.py`: **True**. Quantidade de preditores: **7**.

**Explicitamente fora do modelo:** identificadores, IAN, INDE, Pedra, IPP, fase ideal, variaveis futuras, genero, idade, datas, escola, instituicao, unidade, textos administrativos. (Ex.: IAN é derivado da defasagem; INDE e Pedra compõem-se dos próprios indicadores; gênero só serviria a uma auditoria de equidade e não está nas coortes.)

**Limitações.** Indicadores são medidas institucionais, não medidas diretas de aprendizagem; IDA, IEG e IPV
têm ausências (11 casos no teste, visíveis na seção 16). Fase é um código, e não prova equivalência
curricular entre anos.

## 11. Construção das coortes temporais

**O que foi feito.** Duas coortes com regras de exclusão aplicadas em sequência, sem dupla contagem:
`src/preparacao_coortes.py` produz X (sete colunas), y e uma chave privada de auditoria em arquivos separados.
**Por quê.** Reproduzir o cenário real — treinar no passado, avaliar no futuro. **Resultado.**

In [11]:
etapas = [("Registros na origem", "registros_origem"), ("Já defasados na origem", ("exclusoes_sequenciais", "ja_defasado")),
          ("Fase 8 (regras próprias)", ("exclusoes_sequenciais", "fase_8")),
          ("Fase 9 / não elegível / RA inválido / defasagem indisponível", None),
          ("**Elegíveis na origem**", "elegiveis_origem"), ("Encontrados no ano seguinte", "encontrados"),
          ("Não encontrados (alvo desconhecido)", "nao_encontrados"), ("**Supervisionados (usados no modelo)**", "supervisionados"),
          ("  dos quais alvo = 1", "alvo_1"), ("  dos quais alvo = 0", "alvo_0")]


def valor(coorte, chave):
    if chave is None:
        e = coorte["exclusoes_sequenciais"]
        return e["fase_9"] + e["fase_nao_elegivel"] + e["ra_invalido"] + e["defasagem_indisponivel"]
    if isinstance(chave, tuple):
        return coorte[chave[0]][chave[1]]
    return coorte[chave]


d, t = COR["desenvolvimento"], COR["teste_temporal"]
mostrar(tabela(["Etapa", "Desenvolvimento 2022→2023", "Teste temporal 2023→2024"],
               [[nome, valor(d, ch), valor(t, ch)] for nome, ch in etapas]))
verif = [(nome, ok) for nome, ok in COORTES["validations"].items() if isinstance(ok, bool)]
mostrar("**Validações registradas na geração das coortes:** " + "; ".join(f"{n.replace('_', ' ')}: {o}" for n, o in verif) + ".")
sob = COORTES["summary"]["sobreposicao"]
mostrar(f"**Alunos repetidos:** {sob['teste_com_participacao_desenvolvimento']} das transições do teste pertencem a alunos que também "
        f"estavam no desenvolvimento; {sob['teste_sem_participacao_desenvolvimento']} não. Isso é tratado na análise de robustez (seção 16).")

| Etapa | Desenvolvimento 2022→2023 | Teste temporal 2023→2024 |
| --- | --- | --- |
| Registros na origem | 860 | 1014 |
| Já defasados na origem | 601 | 552 |
| Fase 8 (regras próprias) | 0 | 63 |
| Fase 9 / não elegível / RA inválido / defasagem indisponível | 0 | 0 |
| **Elegíveis na origem** | 259 | 399 |
| Encontrados no ano seguinte | 189 | 311 |
| Não encontrados (alvo desconhecido) | 70 | 88 |
| **Supervisionados (usados no modelo)** | 189 | 311 |
|   dos quais alvo = 1 | 60 | 84 |
|   dos quais alvo = 0 | 129 | 227 |

**Validações registradas na geração das coortes:** coortes serializadas conferidas: True; X sem identificadores: True; X somente variaveis origem: True; fase categorica: True; desconhecidos fora matrizes supervisionadas: True; separacao temporal preservada: True.

**Alunos repetidos:** 104 das transições do teste pertencem a alunos que também estavam no desenvolvimento; 207 não. Isso é tratado na análise de robustez (seção 16).

**Limitações.** Amostra de desenvolvimento pequena (poucas dezenas de eventos), um único corte temporal e
perda de acompanhamento nas duas transições. Alguns indicadores estão ausentes no teste (11 casos), mas
não no desenvolvimento.

## 12. Prevenção de vazamento temporal

**O que foi feito.** Um conjunto de barreiras, cada uma verificável nos artefatos. **Por quê.**
*Vazamento* é o modelo "ver" informação que não teria na prática, o que inflaria o desempenho.
**Resultado.**

| Barreira | Como é garantida |
| --- | --- |
| Só passado como entrada | todos os preditores vêm do registro do ano de origem; o alvo usa apenas a defasagem do destino |
| Sem identificadores | RA e nome ficam em bloco privado de auditoria; não pertencem a X |
| Sem campos derivados do desfecho | IAN, INDE, Pedra, fase ideal e informações do ano seguinte estão excluídos |
| Separação temporal | desenvolvimento e teste em anos distintos, sem embaralhamento |
| Transformações dentro do pipeline | imputação e codificação ajustadas somente na parte de treino de cada dobra |
| Teste aberto uma vez | configuração congelada **antes**; abertura registrada; nova execução só verifica hashes |

In [12]:
lista_x = set(SCHEMA["colunas"])
proibidos = {"ra", "nome", "ian", "inde", "pedra", "ipp", "genero", "idade"}
mostrar(tabela(["Verificação", "Resultado"], [
    ["Nenhum campo proibido entre os preditores", not (lista_x & proibidos)],
    ["X sem identificadores (validação das coortes)", COORTES["validations"]["X_sem_identificadores"]],
    ["X somente com variáveis da origem", COORTES["validations"]["X_somente_variaveis_origem"]],
    ["Alvo desconhecido fora das matrizes supervisionadas", COORTES["validations"]["desconhecidos_fora_matrizes_supervisionadas"]],
    ["Separação temporal preservada", COORTES["validations"]["separacao_temporal_preservada"]],
    ["Configuração congelada em", CONGELADO["congelado_em"]],
    ["Teste temporal aberto em (uma única vez)", ACESSO["aberta_em"]],
    ["Avaliação concluída em", ACESSO["concluida_em"]],
    ["Congelamento anterior à abertura do teste", CONGELADO["congelado_em"] < ACESSO["aberta_em"]],
]))

| Verificação | Resultado |
| --- | --- |
| Nenhum campo proibido entre os preditores | True |
| X sem identificadores (validação das coortes) | True |
| X somente com variáveis da origem | True |
| Alvo desconhecido fora das matrizes supervisionadas | True |
| Separação temporal preservada | True |
| Configuração congelada em | 2026-09-18T14:50:56.482500+00:00 |
| Teste temporal aberto em (uma única vez) | 2026-09-18T14:54:23.641763+00:00 |
| Avaliação concluída em | 2026-09-18T14:56:54.748782+00:00 |
| Congelamento anterior à abertura do teste | True |

**Limitações.** As barreiras protegem contra vazamento *técnico*. Elas não eliminam a repetição de alunos
entre desenvolvimento e teste (medida na seção 16) nem mudanças de instrumento entre anos.

## 13. Modelo oficialmente avaliado

**O que foi feito.** Quatro candidatos — referência de prevalência (Dummy), **regressão logística
regularizada**, árvore rasa e floresta aleatória com complexidade limitada — foram comparados **apenas no
desenvolvimento**, com as mesmas cinco dobras e uma configuração por família fixada antes do teste.
**Por quê.** Com poucos eventos, modelos simples e regularizados reduzem o risco de memorizar ruído.
A regra de escolha, também definida antes, foi a **maior AP** nas probabilidades fora da dobra (OOF). **Resultado.**

In [13]:
linhas = []
for nome in modelagem.ORDER:
    c = MET["comparacao_modelos"][nome]
    m, disp = c["metricas"], c["dispersao"]
    linhas.append([("**" + nome + "**") if nome == MET["modelo"] else nome, num(m["average_precision"]), num(m["roc_auc"]),
                   num(m["brier"]), f'{disp["average_precision"]["media"]:.4f} ± {disp["average_precision"]["dp"]:.4f}',
                   num(c["limiar"])])
mostrar(tabela(["Candidato", "AP OOF", "ROC-AUC OOF", "Brier OOF", "AP por dobra (média ± DP)", "Limiar OOF próprio"], linhas))
mostrar(f"**Selecionado:** `{MET['modelo']}` (regra: {CONGELADO['configuracao']['protocolo']['selecao']})")

cfg = CONGELADO["configuracao"]
hp = cfg["hiperparametros"]
prep_cfg = cfg["protocolo"]["preprocessamento"]
mostrar(tabela(["Componente do pipeline oficial", "Configuração"], [
    ["Algoritmo", cfg["algoritmo"]],
    ["Regularização (C) / solver / iterações máximas", f'{hp["C"]} / {hp["solver"]} / {hp["max_iter"]}'],
    ["Balanceamento artificial", f'{hp["class_weight"]} (nenhum)'],
    ["Semente", cfg["protocolo"]["random_state"]],
    ["Numéricos", "mediana (imputação por dobra) + padronização"],
    ["Fase", prep_cfg["fase"]],
    ["Indicadores de ausência", f'{prep_cfg["indicadores_ausencia"]} — {prep_cfg["justificativa"]}'],
    ["Treino do modelo avaliado", f'{SCHEMA["treino"]} (n={SCHEMA["n"]}, eventos={SCHEMA["eventos"]})'],
    ["Hash do arquivo do modelo (SHA-256)", SCHEMA["modelo_sha256"][:16] + "…"],
]))

| Candidato | AP OOF | ROC-AUC OOF | Brier OOF | AP por dobra (média ± DP) | Limiar OOF próprio |
| --- | --- | --- | --- | --- | --- |
| **logistica** | 0.6649 | 0.8142 | 0.1604 | 0.6874 ± 0.0224 | 0.2670 |
| arvore | 0.5029 | 0.7290 | 0.1949 | 0.5027 ± 0.0858 | 0.2222 |
| floresta | 0.5747 | 0.7665 | 0.1859 | 0.5953 ± 0.0402 | 0.3070 |
| dummy | 0.3161 | 0.4969 | 0.2167 | 0.3175 ± 0.0038 | 0.3158 |

**Selecionado:** `logistica` (regra: Maior AP OOF; empate exato: menor DP de AP nos folds, menor Brier OOF, ordem logistica/arvore/floresta/dummy.)

| Componente do pipeline oficial | Configuração |
| --- | --- |
| Algoritmo | LogisticRegression |
| Regularização (C) / solver / iterações máximas | 1.0 / lbfgs / 2000 |
| Balanceamento artificial | None (nenhum) |
| Semente | 42 |
| Numéricos | mediana (imputação por dobra) + padronização |
| Fase | OneHotEncoder categorias 0..7, sem descarte, handle_unknown=ignore; desconhecida gera vetor zero |
| Indicadores de ausência | False — Nenhuma ausencia observada no desenvolvimento; nao ha padrao aprendivel. |
| Treino do modelo avaliado | 2022->2023 (n=189, eventos=60) |
| Hash do arquivo do modelo (SHA-256) | 88098ab093fab4ec… |

**Limitações.** Com cinco dobras e 60 eventos, a diferença entre candidatos **não é** prova de superioridade
estatística; a escolha segue a regra pré-definida. As métricas internas têm otimismo de seleção e **não
substituem** o teste temporal. Este é o modelo *avaliado*; ele não é um modelo de produção.

## 14. Limiar congelado

**O que foi feito.** O limiar transforma probabilidade em "sinalizar para atenção". Foi escolhido nas
probabilidades fora da dobra do desenvolvimento pela regra: *recall ≥ 0,80; entre os que atendem, maior
precisão; empate: maior recall, depois maior limiar*. Depois **congelado**. **Por quê.** O programa prioriza
não deixar de identificar quem realmente entrará em defasagem (sensibilidade), aceitando alarmes falsos.
**Resultado.**

In [14]:
oof = MET["oof"]["metricas"]
(vn, fp), (fn, vp) = oof["matriz_confusao"]
mostrar(tabela(["Item", "Valor"], [
    ["Regra de decisão", f'`{SCHEMA["regra_classificacao"]}`'],
    ["**Limiar congelado**", f'**{MET["limiar"]:.6f}**'],
    ["Igual no schema, no congelamento e nas métricas", SCHEMA["limiar"] == CONGELADO["configuracao"]["limiar"] == MET["limiar"]],
    ["Regra de escolha", CONGELADO["configuracao"]["protocolo"]["limiar"]],
    ["No desenvolvimento (OOF): precisão / recall / F1", f'{num(oof["precisao"])} / {num(oof["recall"])} / {num(oof["f1"])}'],
    ["No desenvolvimento: sinalizados", f'{oof["previstos_risco"]} de {oof["n"]} ({pct(oof["proporcao_risco"])})'],
    ["Matriz OOF [[VN, FP], [FN, VP]]", f'[[{vn}, {fp}], [{fn}, {vp}]]'],
]))

| Item | Valor |
| --- | --- |
| Regra de decisão | `probabilidade >= limiar` |
| **Limiar congelado** | **0.266967** |
| Igual no schema, no congelamento e nas métricas | True |
| Regra de escolha | p >= limiar; recall >= 0.80; maior precisao; empate: maior recall, depois maior limiar. |
| No desenvolvimento (OOF): precisão / recall / F1 | 0.5326 / 0.8167 / 0.6447 |
| No desenvolvimento: sinalizados | 92 de 189 (48.7%) |
| Matriz OOF [[VN, FP], [FN, VP]] | [[86, 43], [11, 49]] |

**Limitações.** A meta de recall é uma **regra de escolha no desenvolvimento, não uma garantia** para o
próximo ano (veja a seção 15). Políticas alternativas — como selecionar os *N* alunos de maior probabilidade
segundo a capacidade de atendimento — poderão ser estudadas no futuro, mas não alteram retroativamente as
métricas oficiais.

## 15. Métricas de desenvolvimento e avaliação temporal

**O que foi feito.** O modelo treinado só em 2022→2023 foi aplicado **uma única vez** a 2023→2024, com o
limiar congelado; os resultados abaixo são os registrados nessa abertura (não foram recalculados aqui).
**Por quê.** É a estimativa mais honesta de desempenho em dados futuros. **Resultado.**

In [15]:
def linha_metricas(rotulo, m):
    return [rotulo, m["n"], m["eventos"], num(m["average_precision"]), num(m["roc_auc"]), num(m["precisao"]),
            num(m["recall"]), num(m["f1"]), num(m["brier"]), m["previstos_risco"]]


CAB = ["População", "n", "Eventos", "AP", "ROC-AUC", "Precisão", "Recall", "F1", "Brier", "Sinalizados"]
mostrar(tabela(CAB, [linha_metricas("Desenvolvimento (OOF)", DEV), linha_metricas("**Teste temporal**", TESTE)]))
(vn, fp), (fn, vp) = TESTE["matriz_confusao"]
mostrar(f"Prevalência no teste: {pct(TESTE['prevalencia'], 2)}; proporção sinalizada: {pct(TESTE['proporcao_risco'], 2)}. "
        f"Matriz no teste [[VN, FP], [FN, VP]]: `[[{vn}, {fp}], [{fn}, {vp}]]` — "
        f"{vp} dos {TESTE['eventos']} alunos que entraram em defasagem foram sinalizados; {fn} não foram.")

linhas = []
for metrica, rotulo in (("average_precision", "AP"), ("roc_auc", "ROC-AUC"), ("precisao", "Precisão"),
                        ("recall", "Recall"), ("f1", "F1"), ("brier", "Brier (menor é melhor)")):
    i = MET["robustez"]["intervalos"]["completo"][metrica]
    linhas.append([rotulo, num(TESTE[metrica]), f'{i["ic95"][0]:.4f} a {i["ic95"][1]:.4f}',
                   f'{MET["diferenca_temporal_menos_oof"][metrica]:+.4f}', f'{i["validas"]} / {i["invalidas"]}'])
mostrar("**Incerteza no teste** (*bootstrap* percentil, 2.000 reamostragens, semente 42; pipeline e limiar fixos):\n\n"
        + tabela(["Métrica", "Teste", "IC 95%", "Teste − OOF", "Reamostragens válidas / inválidas"], linhas))
mostrar(f"**Leitura:** a capacidade de *ordenar* (AP e ROC-AUC) se manteve próxima do desenvolvimento, mas o **recall caiu de "
        f"{pct(DEV['recall'])} para {pct(TESTE['recall'])}** ({100 * MET['diferenca_temporal_menos_oof']['recall']:+.1f} pontos percentuais), "
        f"e a precisão subiu. Em outras palavras, o limiar escolhido no passado deixou de identificar mais da metade dos casos do ano seguinte.")

| População | n | Eventos | AP | ROC-AUC | Precisão | Recall | F1 | Brier | Sinalizados |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Desenvolvimento (OOF) | 189 | 60 | 0.6649 | 0.8142 | 0.5326 | 0.8167 | 0.6447 | 0.1604 | 92 |
| **Teste temporal** | 311 | 84 | 0.6211 | 0.8044 | 0.6415 | 0.4048 | 0.4964 | 0.1735 | 53 |

Prevalência no teste: 27.01%; proporção sinalizada: 17.04%. Matriz no teste [[VN, FP], [FN, VP]]: `[[208, 19], [50, 34]]` — 34 dos 84 alunos que entraram em defasagem foram sinalizados; 50 não foram.

**Incerteza no teste** (*bootstrap* percentil, 2.000 reamostragens, semente 42; pipeline e limiar fixos):

| Métrica | Teste | IC 95% | Teste − OOF | Reamostragens válidas / inválidas |
| --- | --- | --- | --- | --- |
| AP | 0.6211 | 0.5194 a 0.7250 | -0.0438 | 2000 / 0 |
| ROC-AUC | 0.8044 | 0.7511 a 0.8536 | -0.0098 | 2000 / 0 |
| Precisão | 0.6415 | 0.5094 a 0.7675 | +0.1089 | 2000 / 0 |
| Recall | 0.4048 | 0.3021 a 0.5122 | -0.4119 | 2000 / 0 |
| F1 | 0.4964 | 0.3881 a 0.5974 | -0.1484 | 2000 / 0 |
| Brier (menor é melhor) | 0.1735 | 0.1409 a 0.2088 | +0.0131 | 2000 / 0 |

**Leitura:** a capacidade de *ordenar* (AP e ROC-AUC) se manteve próxima do desenvolvimento, mas o **recall caiu de 81.7% para 40.5%** (-41.2 pontos percentuais), e a precisão subiu. Em outras palavras, o limiar escolhido no passado deixou de identificar mais da metade dos casos do ano seguinte.

**Curvas oficiais de precisão-recall e de calibração** (arquivo versionado, não regenerado aqui):

![Curvas agregadas de precisão-recall e calibração](../reports/curvas_modelagem.png)

*Esquerda:* precisão em função do recall (envelope em 21 pontos fixos); a linha pontilhada é a prevalência
do teste. *Direita:* probabilidade média prevista × fração observada de eventos por faixa; a diagonal é a
calibração ideal. No teste, as faixas ficam **acima** da diagonal: o modelo **subestimou** o risco.

In [16]:
linhas = []
for pop, rotulo in (("oof", "Desenvolvimento (OOF)"), ("temporal", "Teste temporal")):
    for g in MET[pop]["calibracao"]["grupos"]:
        linhas.append([rotulo, g["n"], num(g["probabilidade_media"]), num(g["fracao_eventos"])])
mostrar(tabela(["População", "n na faixa", "Probabilidade média prevista", "Fração observada de eventos"], linhas))

| População | n na faixa | Probabilidade média prevista | Fração observada de eventos |
| --- | --- | --- | --- |
| Desenvolvimento (OOF) | 76 | 0.0965 | 0.0921 |
| Desenvolvimento (OOF) | 51 | 0.2900 | 0.2745 |
| Desenvolvimento (OOF) | 30 | 0.4865 | 0.5333 |
| Desenvolvimento (OOF) | 32 | 0.7249 | 0.7188 |
| Teste temporal | 243 | 0.0539 | 0.1811 |
| Teste temporal | 38 | 0.2860 | 0.4474 |
| Teste temporal | 30 | 0.6030 | 0.7667 |

**Limitações.** Um único corte temporal e 84 eventos: os intervalos são largos e **não** incluem a incerteza
do treino/seleção. As faixas de calibração são descritivas e **não** foram usadas para recalibrar. AP depende
da prevalência e da composição da coorte.

## 16. Robustez, calibração, equidade e análise de erros

**O que foi feito.** Análises **predefinidas** no contrato, todas com o mesmo modelo e limiar (sem reajuste
por subgrupo) e registradas na avaliação oficial. **Por quê.** Verificar se o resultado depende de casos
incompletos, de alunos repetidos, da variável defasagem ou de mudanças na população. Nenhuma foi refeita
neste notebook. **Resultado, por tema:**

### 16.1 Robustez: casos completos e alunos repetidos

In [17]:
nomes = {"completo": "Teste completo", "casos_completos": "Somente casos completos", "sem_repetidos": "Alunos que **não** estavam no desenvolvimento",
         "repetidos": "Alunos que **estavam** no desenvolvimento"}
g = MET["robustez"]["grupos"]
mostrar(tabela(CAB, [linha_metricas(nomes[k], g[k]) for k in nomes]))
d_ic = MET["robustez"]["intervalos"]["diferenca_sem_repetidos_menos_completo"]
mostrar("**Diferença (sem repetidos − teste completo), reamostragem pareada:**\n\n" + tabela(
    ["Métrica", "Diferença pontual", "IC 95%"],
    [[m, f'{MET["robustez"]["diferenca_sem_repetidos_menos_completo"][m]:+.4f}', f'{d_ic[m]["ic95"][0]:.4f} a {d_ic[m]["ic95"][1]:.4f}']
     for m in ("average_precision", "roc_auc", "precisao", "recall", "f1", "brier")]))
mostrar("Leitura: o recall é semelhante entre os recortes e a AP é maior entre os alunos que não estavam no desenvolvimento. "
        f"O grupo de repetidos tem apenas {g['repetidos']['eventos']} eventos, o que torna suas métricas instáveis; "
        "a diferença é descrita, sem atribuição de causa, e não é um teste independente entre populações.")

| População | n | Eventos | AP | ROC-AUC | Precisão | Recall | F1 | Brier | Sinalizados |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Teste completo | 311 | 84 | 0.6211 | 0.8044 | 0.6415 | 0.4048 | 0.4964 | 0.1735 | 53 |
| Somente casos completos | 300 | 84 | 0.6230 | 0.7971 | 0.6415 | 0.4048 | 0.4964 | 0.1798 | 53 |
| Alunos que **não** estavam no desenvolvimento | 207 | 76 | 0.7194 | 0.8007 | 0.7317 | 0.3947 | 0.5128 | 0.2251 | 41 |
| Alunos que **estavam** no desenvolvimento | 104 | 8 | 0.2907 | 0.7513 | 0.3333 | 0.5000 | 0.4000 | 0.0710 | 12 |

**Diferença (sem repetidos − teste completo), reamostragem pareada:**

| Métrica | Diferença pontual | IC 95% |
| --- | --- | --- |
| average_precision | +0.0983 | 0.0376 a 0.1608 |
| roc_auc | -0.0037 | -0.0376 a 0.0313 |
| precisao | +0.0902 | 0.0150 a 0.1777 |
| recall | -0.0100 | -0.0494 a 0.0237 |
| f1 | +0.0165 | -0.0280 a 0.0563 |
| brier | +0.0515 | 0.0312 a 0.0722 |

Leitura: o recall é semelhante entre os recortes e a AP é maior entre os alunos que não estavam no desenvolvimento. O grupo de repetidos tem apenas 8 eventos, o que torna suas métricas instáveis; a diferença é descrita, sem atribuição de causa, e não é um teste independente entre populações.

### 16.2 Sensibilidade: modelo sem a defasagem de origem

In [18]:
s = MET["sensibilidade_sem_defasagem"]
mostrar(tabela(CAB, [linha_metricas("Sem defasagem — OOF", s["oof"]["metricas"]), linha_metricas("Sem defasagem — teste", s["temporal"])])
        + f"\n\nLimiar próprio da sensibilidade: {s['oof']['limiar']:.6f}. Serve só como comparação; **não** substitui o modelo principal.")

| População | n | Eventos | AP | ROC-AUC | Precisão | Recall | F1 | Brier | Sinalizados |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Sem defasagem — OOF | 189 | 60 | 0.6295 | 0.7996 | 0.5161 | 0.8000 | 0.6275 | 0.1671 | 93 |
| Sem defasagem — teste | 311 | 84 | 0.6109 | 0.7939 | 0.6275 | 0.3810 | 0.4741 | 0.1753 | 51 |

Limiar próprio da sensibilidade: 0.267513. Serve só como comparação; **não** substitui o modelo principal.

### 16.3 Mudança de distribuição entre desenvolvimento e teste

In [19]:
dist = MET["robustez"]["distribuicao"]["numericos"]
mostrar(tabela(["Preditor", "Média desenvolvimento", "Média teste", "Diferença padronizada", "Ausentes (dev / teste)"],
               [[k.upper(), num(v["media_desenvolvimento"]), num(v["media_teste"]), f'{v["diferenca_padronizada"]:+.3f}',
                 f'{v["ausentes_desenvolvimento"]} / {v["ausentes_teste"]}'] for k, v in dist.items()]))
mostrar("A maior mudança está no **IPS** (diferença padronizada negativa), seguida do IPV, IEG e IAA. Isso descreve a mudança da população "
        "— não demonstra a causa da queda de recall.")

| Preditor | Média desenvolvimento | Média teste | Diferença padronizada | Ausentes (dev / teste) |
| --- | --- | --- | --- | --- |
| IDA | 6.7841 | 6.9897 | +0.138 | 0 / 11 |
| IEG | 8.5460 | 9.0750 | +0.518 | 0 / 11 |
| IAA | 8.5656 | 7.1730 | -0.516 | 0 / 0 |
| IPS | 6.9487 | 4.9268 | -1.187 | 0 / 0 |
| IPV | 7.5964 | 8.2577 | +0.763 | 0 / 11 |
| DEFASAGEM_ORIGEM | 0.0741 | 0.1222 | +0.139 | 0 / 0 |

A maior mudança está no **IPS** (diferença padronizada negativa), seguida do IPV, IEG e IAA. Isso descreve a mudança da população — não demonstra a causa da queda de recall.

### 16.4 Perda de acompanhamento

In [20]:
perdas = MET["robustez"]["perdas"]
mostrar(f"**{perdas['sem_destino']['n']} elegíveis de 2023 não foram encontrados em 2024.** Nenhum alvo foi atribuído a eles; "
        f"encontrados sem defasagem futura válida: {perdas['encontrados_sem_alvo']}. Perfis na origem:")
linhas = []
for grupo, rotulo in (("sem_destino", "Sem destino"), ("encontrados", "Encontrados")):
    for k, v in perdas[grupo]["numericos"].items():
        linhas.append([rotulo, k.upper(), f'{v["disponiveis"]}', v["ausentes"], num(v["media"]), num(v["mediana"], 3)])
mostrar(tabela(["Grupo", "Preditor", "Disponíveis", "Ausentes", "Média", "Mediana"], linhas))

**88 elegíveis de 2023 não foram encontrados em 2024.** Nenhum alvo foi atribuído a eles; encontrados sem defasagem futura válida: 0. Perfis na origem:

| Grupo | Preditor | Disponíveis | Ausentes | Média | Mediana |
| --- | --- | --- | --- | --- | --- |
| Sem destino | IDA | 88 | 0 | 6.6807 | 7.000 |
| Sem destino | IEG | 88 | 0 | 8.3307 | 8.400 |
| Sem destino | IAA | 88 | 0 | 6.6102 | 8.300 |
| Sem destino | IPS | 87 | 1 | 5.3469 | 5.000 |
| Sem destino | IPV | 88 | 0 | 7.9129 | 8.045 |
| Sem destino | DEFASAGEM_ORIGEM | 88 | 0 | 0.0795 | 0.000 |
| Encontrados | IDA | 300 | 11 | 6.9897 | 7.050 |
| Encontrados | IEG | 300 | 11 | 9.0750 | 9.300 |
| Encontrados | IAA | 311 | 0 | 7.1730 | 8.500 |
| Encontrados | IPS | 311 | 0 | 4.9268 | 5.000 |
| Encontrados | IPV | 300 | 11 | 8.2577 | 8.289 |
| Encontrados | DEFASAGEM_ORIGEM | 311 | 0 | 0.1222 | 0.000 |

### 16.5 Equidade descritiva por fase (limiar fixo)

Gênero **não** integra as coortes aprovadas; portanto a auditoria de equidade por gênero foi registrada como
*indisponível*. Por fase, métricas só são divulgadas com pelo menos 30 observações e 5 eventos e 5 não eventos.

In [21]:
eq_g = MET["robustez"]["equidade_genero"]
mostrar(f"Equidade por gênero: **{eq_g['status']}** — {eq_g['motivo']}")
linhas, suprimidas = [], []
for fase, item in MET["robustez"]["equidade_fase"].items():
    if item["status"] == "estimado":
        linhas.append(linha_metricas(f"Fase {fase}", item["metricas"]))
    else:
        suprimidas.append(fase)
mostrar(tabela(CAB, linhas))
mostrar(f"Fases com métricas **suprimidas** por tamanho insuficiente: {', '.join(suprimidas)}. "
        "Diferenças entre fases são descritivas: refletem também prevalências e tamanhos muito distintos, e **não** demonstram equidade nem inequidade causal.")

Equidade por gênero: **indisponivel** — As coortes aprovadas nao incluem genero, nem nos metadados. Nao se incorpora outra base. Inconsistencias cadastrais ja documentadas limitariam a interpretacao.

| População | n | Eventos | AP | ROC-AUC | Precisão | Recall | F1 | Brier | Sinalizados |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Fase 0 | 81 | 53 | 0.8462 | 0.7702 | 0.8462 | 0.4151 | 0.5570 | 0.3870 | 26 |
| Fase 1 | 33 | 16 | 0.7570 | 0.7206 | 0.5882 | 0.6250 | 0.6061 | 0.2491 | 17 |
| Fase 3 | 51 | 6 | 0.1455 | 0.5667 | não estimável | 0.0000 | 0.0000 | 0.1124 | 0 |

Fases com métricas **suprimidas** por tamanho insuficiente: 2, 4, 5, 6, 7. Diferenças entre fases são descritivas: refletem também prevalências e tamanhos muito distintos, e **não** demonstram equidade nem inequidade causal.

### 16.6 Análise de erros: falsos positivos e falsos negativos (perfis agregados)

In [22]:
erros = MET["robustez"]["erros"]
linhas = []
for grupo, rotulo in (("falsos_positivos", "Falsos positivos"), ("falsos_negativos", "Falsos negativos")):
    for k, v in erros[grupo]["numericos"].items():
        linhas.append([rotulo, erros[grupo]["n"], k.upper(), num(v["media"]), num(v["mediana"], 3)])
mostrar(tabela(["Grupo", "n", "Preditor", "Média", "Mediana"], linhas))
mostrar("Os falsos negativos concentram-se em alunos com **IPS mais baixo** que os falsos positivos. Falso positivo **não** significa "
        "ausência de necessidade pedagógica: o alvo cobre apenas a entrada em defasagem observada.")

| Grupo | n | Preditor | Média | Mediana |
| --- | --- | --- | --- | --- |
| Falsos positivos | 19 | IDA | 6.9579 | 7.300 |
| Falsos positivos | 19 | IEG | 8.7105 | 8.800 |
| Falsos positivos | 19 | IAA | 5.7842 | 8.500 |
| Falsos positivos | 19 | IPS | 7.1889 | 7.520 |
| Falsos positivos | 19 | IPV | 7.7286 | 7.783 |
| Falsos positivos | 19 | DEFASAGEM_ORIGEM | 0.0000 | 0.000 |
| Falsos negativos | 50 | IDA | 6.8240 | 6.700 |
| Falsos negativos | 50 | IEG | 8.9120 | 9.050 |
| Falsos negativos | 50 | IAA | 7.0900 | 9.000 |
| Falsos negativos | 50 | IPS | 4.8674 | 3.770 |
| Falsos negativos | 50 | IPV | 8.3329 | 8.339 |
| Falsos negativos | 50 | DEFASAGEM_ORIGEM | 0.0200 | 0.000 |

Os falsos negativos concentram-se em alunos com **IPS mais baixo** que os falsos positivos. Falso positivo **não** significa ausência de necessidade pedagógica: o alvo cobre apenas a entrada em defasagem observada.

### 16.7 Interpretabilidade (coeficientes do modelo avaliado)

In [23]:
coefs = MET["interpretabilidade"]["valores"]
ordem = sorted(coefs.items(), key=lambda kv: -abs(kv[1]))
mostrar(tabela(["Atributo transformado", "Coeficiente (log-odds)"], [[k.replace("__", " › "), f"{v:+.4f}"] for k, v in ordem]))
mostrar(MET["interpretabilidade"]["nota"] + " Coeficientes descrevem **associações condicionais** ajustadas pelas demais variáveis; "
        "não são efeitos causais e podem ser instáveis com poucos eventos e indicadores correlacionados.")

| Atributo transformado | Coeficiente (log-odds) |
| --- | --- |
| fase › fase_origem_3 | -1.1449 |
| numericos › ipv | -1.1333 |
| fase › fase_origem_1 | +1.0620 |
| numericos › defasagem_origem | -0.9485 |
| fase › fase_origem_0 | +0.6882 |
| fase › fase_origem_7 | -0.5959 |
| numericos › ips | +0.4740 |
| fase › fase_origem_6 | -0.4280 |
| numericos › ieg | +0.4051 |
| fase › fase_origem_2 | +0.2712 |
| numericos › ida | -0.2439 |
| fase › fase_origem_5 | +0.1856 |
| fase › fase_origem_4 | -0.0401 |
| numericos › iaa | -0.0225 |

Numericos por um desvio padrao do desenvolvimento; fase one-hot sem referencia descartada, contrastes entre fases sao diferencas entre coeficientes. Associacoes condicionais, nao causais. Coeficientes descrevem **associações condicionais** ajustadas pelas demais variáveis; não são efeitos causais e podem ser instáveis com poucos eventos e indicadores correlacionados.

**Limitações da seção 16.** Subgrupos pequenos, intervalos amplos, indicadores correlacionados e comparações
múltiplas exploratórias. Nada aqui reajusta o modelo, o limiar ou a calibração.

## 17. Respostas às 11 perguntas de negócio

**O que foi feito.** As respostas vêm do relatório oficial `reports/relatorio_analises_negocio.md`
(gerado por `src/analises_negocio.py` e conferido por hash). Abaixo, o notebook apenas **lê** e organiza a
*matriz de síntese* — pergunta, evidência principal, limitação e recomendação — sem alterar texto ou números.
**Por quê.** Manter uma única fonte de verdade. **Limitação comum a todas:** análises observacionais,
sem grupo de controle; associações **não** são causas; totais anuais contam registros, não alunos únicos.

In [24]:
import re

texto = (ROOT / analises.REPORT).read_text(encoding="utf-8")
perguntas = dict((int(n), t) for n, t in re.findall(r"^## (\d+)\. (.+)$", texto, flags=re.M))
matriz = texto.split("## Matriz de síntese", 1)[1].split("## Fontes e reprodução", 1)[0]
linhas_matriz = {}
for linha in matriz.splitlines():
    if linha.startswith("| ") and linha[2].isdigit():
        celulas = [c.strip().replace("\\|", "|") for c in re.split(r"(?<!\\)\|", linha.strip().strip("|"))]
        linhas_matriz[int(celulas[0])] = celulas
assert sorted(perguntas) == sorted(linhas_matriz) == list(range(1, 12)), "Relatório sem as 11 perguntas"
assert len(NEGOCIO["perguntas"]) == 11
for n in range(1, 12):
    _, evidencia, conclusao, limitacao, recomendacao = linhas_matriz[n]
    mostrar(f"**Pergunta {n} — {perguntas[n]}**\n\n- **Evidência principal:** {evidencia}\n- **Como ler:** {conclusao}\n"
            f"- **Limitação:** {limitacao}\n- **Recomendação do relatório:** {recomendacao}")

**Pergunta 1 — Qual é o perfil de defasagem e como evolui?**

- **Evidência principal:** 2022: 66.6% moderada e 3.3% severa; 2023: 53.1% moderada e 1.4% severa. Anos com detalhamento suprimido não autorizam concluir ausência de defasagem severa.
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** IAN incorpora D; sua associação com adequação é matemática. Divergências são sinalizadas, sem correção. Mudanças de composição impedem inferir evolução individual pelas proporções.
- **Recomendação do relatório:** Priorizar acompanhamento pedagógico da defasagem e revisar divergências na origem, mantendo o valor registrado.

**Pergunta 2 — O IDA melhora, permanece estável ou cai entre fases e anos?**

- **Evidência principal:** IDA médio anual: 2022: 6.09; 2023: 6.66; 2024: 6.35. Nos pares: 2022→2023: mudança média +0.12 (n=574); 2023→2024: mudança média -0.48 (n=682).
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** Fases agregadas misturam anos e estudantes repetidos. Códigos não provam equivalência curricular; dispersão não é intervalo de confiança.
- **Recomendação do relatório:** Monitorar IDA e cobertura por fase, distinguindo composição da turma e progresso dos acompanhados.

**Pergunta 3 — Qual a associação de IEG com IDA e IPV?**

- **Evidência principal:** ajustado_ano / ieg x ida: ρ=0.49 (moderada, n=2852); ajustado_ano / ieg x ipv: ρ=0.52 (moderada, n=2852).
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** Associação contemporânea pode refletir contexto comum e mecanismos de avaliação.
- **Recomendação do relatório:** Acompanhar engajamento junto ao desempenho, sem usar correlação como efeito de intervenção.

**Pergunta 4 — A autoavaliação IAA é coerente com IDA e IEG?**

- **Evidência principal:** ajustado_ano / iaa x ida: ρ=0.16 (fraca, n=2851); ajustado_ano / iaa x ieg: ρ=0.21 (fraca, n=2852). Coerência é descrita por ordenação e diferença assinada IAA − indicador; não há classificação de alunos como coerentes/incoerentes.
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** Indicadores não medem o mesmo construto; diferença de pontos não é erro de percepção ou diagnóstico.
- **Recomendação do relatório:** Usar discrepâncias como tema de escuta pedagógica e revisar a comparabilidade dos instrumentos.

**Pergunta 5 — Existem padrões de IPS que antecedem quedas futuras de IDA ou IEG?**

- **Evidência principal:** 2022→2023, ΔIDA: ρ=0.03; 2022→2023, ΔIEG: ρ=-0.02; 2023→2024, ΔIDA: ρ=0.03; 2023→2024, ΔIEG: ρ=0.02. Os cortes complementam a mudança contínua e não definem queda clinicamente relevante. As correlações contínuas próximas de zero não sustentam um padrão monotônico útil de antecedência nesta base; isso não exclui relações não lineares ou dependentes do contexto.
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** Antecedência temporal não estabelece causa; regressão à média, cobertura, contexto e instrumentos podem explicar mudanças. Cortes são descritivos.
- **Recomendação do relatório:** Acompanhar IPS e mudanças futuras em conjunto, registrar alterações de instrumento e não criar triagem automática pelo IPS.

...

**Pergunta 7 — Quais indicadores estão mais associados ao IPV?**

- **Evidência principal:** 2022: maior |ρ| contemporâneo em IDA; 2023: maior |ρ| contemporâneo em IDA; 2024: maior |ρ| contemporâneo em IPP. O ranking é descritivo e pode mudar entre anos e no horizonte futuro.
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** As correlações não isolam efeitos próprios dos indicadores nem ajustam todas as diferenças por fase. IPV também integra o INDE.
- **Recomendação do relatório:** Monitorar os indicadores associados em conjunto e avaliar mudanças de instrumento antes de interpretar tendências.

**Pergunta 8 — Quais combinações de IDA, IEG, IPS e IPP se associam a maior INDE?**

- **Evidência principal:** 2022: maior média publicável 7.96, perfil IDA alto / IEG alto / IPS alto; 2023: maior média publicável 8.25, perfil IDA alto / IEG alto / IPS alto / IPP baixo; 2024: maior média publicável 8.40, perfil IDA alto / IEG alto / IPS alto / IPP alto.
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** Circularidade: INDE combina esses indicadores com IAN, IAA e IPV; pesos/aplicabilidade diferem por fase. Não se recalcula a fórmula, nem se interpreta associação como descoberta causal. Perfis anuais não são diretamente equivalentes.
- **Recomendação do relatório:** Usar perfis para descrição multidimensional; evitar priorizar um componente apenas por sua associação matemática ao INDE.

**Pergunta 9 — Como o modelo avaliado apoia a previsão de risco?**

- **Evidência principal:** Recall cai de 81.7% no OOF para 40.5% no teste (-41.2 pontos percentuais). A meta interna não se mantém no ano seguinte.
- **Como ler:** Discriminação temporal não garante sensibilidade operacional.
- **Limitação:** Desenvolvimento pequeno, único teste temporal, perdas e sobreposição de estudantes. OOF após seleção tem otimismo. O alvo é entrada em defasagem entre elegíveis, não qualquer queda de desempenho.
- **Recomendação do relatório:** Usar como apoio à revisão humana, monitorando cobertura e falsos negativos. Modelo operacional futuro terá identificação própria; não foi criado nesta etapa.

**Pergunta 10 — O que os indicadores mostram sobre evolução nas Pedras?**

- **Evidência principal:** 2022→2023: melhoria de Pedra 24.4%, piora 24.4%; 2023→2024: melhoria de Pedra 24.2%, piora 25.8%.
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** Sem controle ou contrafactual, evolução não confirma impacto do programa. Pedra deriva do desempenho/INDE; faixas documentais divergem. Matriz detalhada pode ser suprimida por células pequenas.
- **Recomendação do relatório:** Acompanhar permanência, regressões e avanços junto à cobertura, sem confundir Pedra com fase escolar.

**Pergunta 11 — Quais insights adicionais orientam coleta e monitoramento?**

- **Evidência principal:** 2022→2023: 70/259 elegíveis sem correspondência futura; 2023→2024: 88/399 elegíveis sem correspondência futura. Maior deslocamento padronizado absoluto entre coortes em IPS. Em 2024, as médias publicáveis de IDA variam de 5.35 na fase 3 a 7.32 na fase 0, indicando heterogeneidade descritiva, sem comparação causal entre fases.
- **Como ler:** Evolução/associação observada; sem inferência causal.
- **Limitação:** Ausência de correspondência não prova evasão, sucesso ou fracasso. Mudanças de distribuição não demonstram a causa da queda de recall. Indicadores e composição podem ter mudado.
- **Recomendação do relatório:** Registrar motivo de saída e calendário de avaliação; padronizar instrumentos e definições por fase; monitorar disponibilidade e perdas a cada ciclo. Revisar casos com a equipe e planejar validação prospectiva separada, sem ajustar retroativamente o modelo.

## 18. Apresentação dos dez gráficos agregados

**O que foi feito.** Os dez gráficos oficiais foram gerados pela etapa de análises e são **referenciados**
(não reproduzidos) a partir de `reports/figures/`; assim, não há imagens duplicadas dentro do notebook.
**Por quê.** Preservar exatamente o que foi validado e evitar divergência entre notebook e relatório.
Todos os gráficos exibem apenas agregados, com supressão de grupos pequenos. Abaixo, a verificação de
existência e integridade e, em seguida, cada figura com sua leitura.

In [25]:
linhas = []
for figura in NEGOCIO["figuras"]:
    caminho = ROOT / figura
    linhas.append([Path(figura).name, caminho.is_file(), rastro.file_matches_sha256(caminho, NEGOCIO["output_hashes"][figura])])
mostrar(tabela(["Figura", "Existe", "Hash confere com o registro oficial"], linhas))
assert len(linhas) == 10 and all(l[1] and l[2] for l in linhas), "Figuras oficiais ausentes ou alteradas"

| Figura | Existe | Hash confere com o registro oficial |
| --- | --- | --- |
| 01_defasagem.png | True | True |
| 02_ida.png | True | True |
| 03_associacoes.png | True | True |
| 04_ips.png | True | True |
| 05_ipp.png | True | True |
| 06_ipv.png | True | True |
| 07_inde.png | True | True |
| 08_pedras.png | True | True |
| 09_modelo.png | True | True |
| 10_insights.png | True | True |

**Gráfico 1 — Composição anual da defasagem**

![1 — Composição anual da defasagem](../reports/figures/01_defasagem.png)

*Leitura:* Proporção de registros sem defasagem, com defasagem moderada e severa em 2022 e 2023. Em 2024 o detalhamento é suprimido por conter célula pequena. Mostra composições anuais, não a trajetória de cada aluno.

**Gráfico 2 — IDA por ano e por fase**

![2 — IDA por ano e por fase](../reports/figures/02_ida.png)

*Leitura:* À esquerda, média e desvio padrão do IDA por ano (as barras são dispersão, não intervalo de confiança); à direita, IDA médio por fase e ano. Fase é código educacional, não Pedra.

**Gráfico 3 — Engajamento e autoavaliação**

![3 — Engajamento e autoavaliação](../reports/figures/03_associacoes.png)

*Leitura:* Correlações de Spearman entre IEG × IDA, IEG × IPV, IAA × IDA e IAA × IEG, por ano e ajustadas por ano. O engajamento se associa moderadamente ao desempenho e ao ponto de virada; a autoavaliação, fracamente.

**Gráfico 4 — IPS de origem e mudança futura de IDA e IEG**

![4 — IPS de origem e mudança futura de IDA e IEG](../reports/figures/04_ips.png)

*Leitura:* Mudança média (destino − origem) de IDA e IEG segundo o IPS de origem ficar abaixo ou na mediana/acima. As correlações contínuas são próximas de zero; não há padrão que sustente triagem pelo IPS.

**Gráfico 5 — IPP médio por adequação registrada**

![5 — IPP médio por adequação registrada](../reports/figures/05_ipp.png)

*Leitura:* IPP médio por categoria de defasagem em 2023 e 2024 (IPP não existe em 2022). Barras omitidas foram suprimidas por privacidade. Não é diagnóstico.

**Gráfico 6 — Associações contemporâneas do IPV**

![6 — Associações contemporâneas do IPV](../reports/figures/06_ipv.png)

*Leitura:* Correlações do IPV com IDA, IEG, IAA, IPS e IPP por ano. Descritivo; IPV também compõe o INDE e nada indica influência causal.

**Gráfico 7 — Perfis de indicadores e INDE médio**

![7 — Perfis de indicadores e INDE médio](../reports/figures/07_inde.png)

*Leitura:* Os dois perfis de menor e de maior INDE médio por ano (↑ na mediana ou acima; ↓ abaixo da mediana anual). Relação em parte matemática: o INDE é composto por esses indicadores.

**Gráfico 8 — Pedras: IDA médio e mudança entre anos**

![8 — Pedras: IDA médio e mudança entre anos](../reports/figures/08_pedras.png)

*Leitura:* À esquerda, IDA médio por Pedra observada; à direita, percentual de melhoria, estabilidade e piora de Pedra nos alunos correspondidos. Pedra é faixa de desempenho, não fase; sem grupo de controle, não mede impacto do programa.

**Gráfico 9 — Métricas oficiais do modelo avaliado**

![9 — Métricas oficiais do modelo avaliado](../reports/figures/09_modelo.png)

*Leitura:* AP, ROC-AUC, precisão, recall, F1 e Brier no desenvolvimento (OOF) e no teste temporal, com o limiar congelado. Destaca a queda do recall no teste. Brier: menor é melhor.

**Gráfico 10 — Disponibilidade de indicadores e mudança entre coortes**

![10 — Disponibilidade de indicadores e mudança entre coortes](../reports/figures/10_insights.png)

*Leitura:* À esquerda, disponibilidade dos indicadores por ano (a queda do IPP em 2022 é estrutural); à direita, diferença padronizada de cada preditor entre teste e desenvolvimento. Cobertura não é desempenho.

**Limitações dos gráficos.** Escalas e cortes seguem os relatórios oficiais; rótulos indicam associação, nunca
efeito. Números exatos de cada figura estão em `reports/relatorio_analises_negocio.md` e
`reports/metricas_analises_negocio.json`.

## 19. Limitações metodológicas e operacionais

**O que foi feito.** Consolidação das limitações já registradas nos relatórios oficiais. **Por quê.** Evitar
conclusões maiores do que as evidências. **Resultado.**

In [26]:
ic = MET["robustez"]["intervalos"]["completo"]
mostrar(f'''
**Metodológicas**
- **Amostra pequena:** {DEV["eventos"]} eventos no desenvolvimento e {TESTE["eventos"]} no teste; um único corte temporal.
- **Otimismo de seleção:** as métricas OOF vêm das mesmas probabilidades usadas para escolher modelo e limiar.
- **Intervalos incompletos:** o *bootstrap* condiciona ao modelo treinado (não inclui a incerteza do treino nem a correlação institucional).
- **Alunos repetidos:** {sob["teste_com_participacao_desenvolvimento"]} transições do teste pertencem a alunos que estavam no desenvolvimento.
- **Perda de acompanhamento:** {COR["desenvolvimento"]["nao_encontrados"]} e {COR["teste_temporal"]["nao_encontrados"]} elegíveis sem desfecho; ausência na base não é sucesso nem fracasso.
- **Mudança de população:** diferenças padronizadas relevantes (p. ex., IPS) e ausências que aparecem só no teste.
- **Definições pendentes:** equivalência curricular entre anos, fase 9, INCLUIR, faixas documentais de Pedra/INDE.
- **Escopo:** fases 8 e 9 e alunos já defasados não são cobertos pelo modelo; gênero não integra as coortes (sem auditoria de equidade por gênero).
- **Observacional:** nenhuma análise identifica causalidade ou efeito do programa.

**Operacionais**
- O recall no teste ({pct(TESTE["recall"])}; IC95% {ic["recall"]["ic95"][0]:.3f} a {ic["recall"]["ic95"][1]:.3f}) mostra que o limiar atual **não captura** a maioria dos casos; a precisão é {pct(TESTE["precisao"])}.
- Probabilidades **não** são diagnóstico; a calibração no teste subestimou o risco.
- O **modelo operacional** (retreinado com as duas transições) **não existe**; o artefato atual é o modelo *avaliado* e não deve ser confundido com produção.
- Qualquer uso exige revisão humana, monitoramento de cobertura e de falsos negativos e validação prospectiva separada.
''')


**Metodológicas**
- **Amostra pequena:** 60 eventos no desenvolvimento e 84 no teste; um único corte temporal.
- **Otimismo de seleção:** as métricas OOF vêm das mesmas probabilidades usadas para escolher modelo e limiar.
- **Intervalos incompletos:** o *bootstrap* condiciona ao modelo treinado (não inclui a incerteza do treino nem a correlação institucional).
- **Alunos repetidos:** 104 transições do teste pertencem a alunos que estavam no desenvolvimento.
- **Perda de acompanhamento:** 70 e 88 elegíveis sem desfecho; ausência na base não é sucesso nem fracasso.
- **Mudança de população:** diferenças padronizadas relevantes (p. ex., IPS) e ausências que aparecem só no teste.
- **Definições pendentes:** equivalência curricular entre anos, fase 9, INCLUIR, faixas documentais de Pedra/INDE.
- **Escopo:** fases 8 e 9 e alunos já defasados não são cobertos pelo modelo; gênero não integra as coortes (sem auditoria de equidade por gênero).
- **Observacional:** nenhuma análise identifica causalidade ou efeito do programa.

**Operacionais**
- O recall no teste (40.5%; IC95% 0.302 a 0.512) mostra que o limiar atual **não captura** a maioria dos casos; a precisão é 64.2%.
- Probabilidades **não** são diagnóstico; a calibração no teste subestimou o risco.
- O **modelo operacional** (retreinado com as duas transições) **não existe**; o artefato atual é o modelo *avaliado* e não deve ser confundido com produção.
- Qualquer uso exige revisão humana, monitoramento de cobertura e de falsos negativos e validação prospectiva separada.


## 20. Considerações de privacidade e ética

**O que foi feito.** Tratamento dos dados como sensíveis: os alunos são, em sua maioria, crianças e jovens
atendidos por uma instituição social. **Princípios aplicados neste trabalho:**

- **Minimização:** identificadores (RA, nome) e campos cadastrais (datas, escola) ficam fora do modelo e dos documentos públicos.
- **Agregação e supressão:** perfis com menos de 10 observações são suprimidos; métricas por grupo exigem tamanho mínimo.
- **Separação física:** originais e derivados individuais em diretórios privados e não versionados, com verificação automática antes de gravar.
- **Sem ligação por nome:** a correspondência entre anos usa somente RA válido e único.
- **Transparência:** ausências, erros e decisões documentados; nenhuma imputação silenciosa; resultados negativos (queda do recall) informados.
- **Uso responsável:** o modelo apoia a priorização e **não decide** exclusão, atendimento ou avaliação de nenhum aluno; falsos positivos e negativos têm custos humanos.
- **Equidade:** a ausência de gênero nas coortes impede essa auditoria; recomenda-se incluí-la em versões futuras com salvaguardas adequadas.
- **Não estigmatização:** "risco" descreve uma probabilidade estatística de entrada em defasagem, não uma característica da criança.

In [27]:
verif = [
    ["Este notebook exibe amostras de linhas ou registros individuais das bases", "não (somente agregados dos artefatos oficiais)"],
    ["Dados individuais carregados neste notebook", "nenhum (na validação opcional, os arquivos privados são apenas conferidos por hash, sem leitura de registros)"],
    ["Relatórios públicos sem caminhos locais/identificadores (`public_text_issues`)",
     all(not rastro.public_text_issues((ROOT / a).read_text(encoding="utf-8")) for a in (analises.REPORT, modelagem.REPORT))],
    ["Equidade por gênero", MET["robustez"]["equidade_genero"]["status"]],
]
mostrar(tabela(["Verificação de privacidade", "Resultado"], verif))

| Verificação de privacidade | Resultado |
| --- | --- |
| Este notebook exibe amostras de linhas ou registros individuais das bases | não (somente agregados dos artefatos oficiais) |
| Dados individuais carregados neste notebook | nenhum (na validação opcional, os arquivos privados são apenas conferidos por hash, sem leitura de registros) |
| Relatórios públicos sem caminhos locais/identificadores (`public_text_issues`) | True |
| Equidade por gênero | indisponivel |

**Limitações.** Agregação reduz, não elimina, o risco de reidentificação; a divulgação externa deve manter apenas
resultados agregados, com os cuidados institucionais e legais aplicáveis a dados de menores.

## 21. Conclusões e possíveis impactos

**O que foi feito.** Síntese do que as evidências sustentam — e do que não sustentam. **Resultado.**

In [28]:
mostrar(f'''
**O que os dados sustentam**
1. **Há sinal preditivo real, mas modesto.** No teste temporal, AP {TESTE["average_precision"]:.3f} (base: prevalência {pct(TESTE["prevalencia"])}) e
   ROC-AUC {TESTE["roc_auc"]:.3f} indicam que o modelo ordena o risco melhor que o acaso.
2. **A regra de decisão não se sustentou.** O limiar congelado, escolhido para recall de {pct(DEV["recall"])}, identificou {pct(TESTE["recall"])} dos casos do ano seguinte;
   com {TESTE["previstos_risco"]} sinalizados, {pct(TESTE["precisao"])} eram de fato casos.
3. **O modelo subestimou o risco no ano seguinte** (calibração acima da diagonal), e a população mudou entre os anos (IPS em especial).
4. **Associações nas análises de negócio:** engajamento (IEG) associa-se moderadamente a desempenho (IDA) e ponto de virada (IPV);
   a autoavaliação (IAA) associa-se fracamente; o IPS não antecipou mudanças futuras de IDA/IEG nesta base.
5. **Qualidade e cobertura dos dados importam:** erros de planilha, ausência estrutural do IPP em 2022 e perda de acompanhamento limitam o que se pode afirmar.

**O que os dados NÃO sustentam:** causalidade, efeito do programa, desempenho operacional garantido, equidade entre gêneros ou diagnóstico individual.

**Possíveis impactos (condicionais a validação adicional)**
- **Apoio à triagem humana:** lista de atenção priorizada, sempre revisada pela equipe pedagógica, tolerando falsos alarmes.
- **Monitoramento:** acompanhar recall, cobertura e falsos negativos a cada ciclo; reavaliar o limiar em processo separado e prospectivo (sem alterar as métricas oficiais).
- **Coleta de dados:** registrar motivo de saída, padronizar instrumentos e definições por fase, manter calendário de avaliação — recomendações do relatório de análises.
- **Próximos passos do plano:** definição de um modelo operacional próprio (com identificação e métricas separadas), aplicação, apresentação e vídeo (`docs/TASKS.md`).
''')


**O que os dados sustentam**
1. **Há sinal preditivo real, mas modesto.** No teste temporal, AP 0.621 (base: prevalência 27.0%) e
   ROC-AUC 0.804 indicam que o modelo ordena o risco melhor que o acaso.
2. **A regra de decisão não se sustentou.** O limiar congelado, escolhido para recall de 81.7%, identificou 40.5% dos casos do ano seguinte;
   com 53 sinalizados, 64.2% eram de fato casos.
3. **O modelo subestimou o risco no ano seguinte** (calibração acima da diagonal), e a população mudou entre os anos (IPS em especial).
4. **Associações nas análises de negócio:** engajamento (IEG) associa-se moderadamente a desempenho (IDA) e ponto de virada (IPV);
   a autoavaliação (IAA) associa-se fracamente; o IPS não antecipou mudanças futuras de IDA/IEG nesta base.
5. **Qualidade e cobertura dos dados importam:** erros de planilha, ausência estrutural do IPP em 2022 e perda de acompanhamento limitam o que se pode afirmar.

**O que os dados NÃO sustentam:** causalidade, efeito do programa, desempenho operacional garantido, equidade entre gêneros ou diagnóstico individual.

**Possíveis impactos (condicionais a validação adicional)**
- **Apoio à triagem humana:** lista de atenção priorizada, sempre revisada pela equipe pedagógica, tolerando falsos alarmes.
- **Monitoramento:** acompanhar recall, cobertura e falsos negativos a cada ciclo; reavaliar o limiar em processo separado e prospectivo (sem alterar as métricas oficiais).
- **Coleta de dados:** registrar motivo de saída, padronizar instrumentos e definições por fase, manter calendário de avaliação — recomendações do relatório de análises.
- **Próximos passos do plano:** definição de um modelo operacional próprio (com identificação e métricas separadas), aplicação, apresentação e vídeo (`docs/TASKS.md`).


## 22. Referências aos artefatos e documentos do projeto

**O que foi feito.** Índice dos artefatos usados, com verificação de existência e, quando o projeto registra o
hash, da integridade. Caminhos relativos à raiz do repositório.

In [29]:
registros = {**ACESSO["output_hashes"], **NEGOCIO["output_hashes"], **CONGELADO["configuracao"]["input_hashes"]}
grupos = [
    ("Governança e método", ["docs/TASKS.md", "docs/contrato_metodologico.md", "docs/registro_decisoes.md", "docs/status_projeto.md",
                             "docs/evidencias_documentais.md", "docs/mapa_campos.md", "docs/inventario_fontes.md", "docs/revisao_auditoria.md"]),
    ("Auditoria e preparação", ["artifacts_meta.json", "reports/relatorio_auditoria_inicial.md", "reports/relatorio_preparacao_inicial.md",
                                "reports/metadados_preparacao.json"]),
    ("Coortes", ["reports/relatorio_coortes_modelagem.md", "reports/metadados_coortes.json"]),
    ("Modelo e avaliação (congelados)", ["artifacts/configuracao_congelada.json", "artifacts/schema_modelo.json", "artifacts/modelo_avaliado.joblib",
                                         "artifacts/avaliacao_temporal.json", "reports/metricas_modelagem.json", "reports/relatorio_modelagem.md",
                                         "reports/curvas_modelagem.png"]),
    ("Análises de negócio", ["reports/relatorio_analises_negocio.md", "reports/metricas_analises_negocio.json", "reports/verificacao_final.md"]
     + list(NEGOCIO["figuras"])),
    ("Código reutilizado neste notebook", ["src/modelagem.py", "src/preparacao_coortes.py", "src/analises_negocio.py", "src/rastreabilidade.py",
                                           "src/dados_pede.py", "src/relatorio_modelagem.py", "src/relatorio_analises.py"]),
]
linhas = []
for grupo, arquivos in grupos:
    for arq in arquivos:
        existe = (ROOT / arq).is_file()
        if arq in registros and existe:
            integridade = "hash confere" if rastro.file_matches_sha256(ROOT / arq, registros[arq]) else "HASH DIVERGE"
        else:
            integridade = "—" if existe else "ausente"
        linhas.append([grupo, f"[`{arq}`](../{arq})", "sim" if existe else "não", integridade])
mostrar(tabela(["Grupo", "Arquivo (link relativo)", "Existe", "Integridade registrada"], linhas))
mostrar("Documentos de referência técnica citados nos relatórios: [imputação em pipelines](https://scikit-learn.org/stable/modules/impute.html) "
        "e [métricas de classificação](https://scikit-learn.org/stable/modules/model_evaluation.html).")

| Grupo | Arquivo (link relativo) | Existe | Integridade registrada |
| --- | --- | --- | --- |
| Governança e método | [`docs/TASKS.md`](../docs/TASKS.md) | sim | — |
| Governança e método | [`docs/contrato_metodologico.md`](../docs/contrato_metodologico.md) | sim | — |
| Governança e método | [`docs/registro_decisoes.md`](../docs/registro_decisoes.md) | sim | — |
| Governança e método | [`docs/status_projeto.md`](../docs/status_projeto.md) | sim | — |
| Governança e método | [`docs/evidencias_documentais.md`](../docs/evidencias_documentais.md) | sim | — |
| Governança e método | [`docs/mapa_campos.md`](../docs/mapa_campos.md) | sim | — |
| Governança e método | [`docs/inventario_fontes.md`](../docs/inventario_fontes.md) | sim | — |
| Governança e método | [`docs/revisao_auditoria.md`](../docs/revisao_auditoria.md) | sim | — |
| Auditoria e preparação | [`artifacts_meta.json`](../artifacts_meta.json) | sim | — |
| Auditoria e preparação | [`reports/relatorio_auditoria_inicial.md`](../reports/relatorio_auditoria_inicial.md) | sim | — |
| Auditoria e preparação | [`reports/relatorio_preparacao_inicial.md`](../reports/relatorio_preparacao_inicial.md) | sim | — |
| Auditoria e preparação | [`reports/metadados_preparacao.json`](../reports/metadados_preparacao.json) | sim | — |
| Coortes | [`reports/relatorio_coortes_modelagem.md`](../reports/relatorio_coortes_modelagem.md) | sim | — |
| Coortes | [`reports/metadados_coortes.json`](../reports/metadados_coortes.json) | sim | — |
| Modelo e avaliação (congelados) | [`artifacts/configuracao_congelada.json`](../artifacts/configuracao_congelada.json) | sim | — |
| Modelo e avaliação (congelados) | [`artifacts/schema_modelo.json`](../artifacts/schema_modelo.json) | sim | hash confere |
| Modelo e avaliação (congelados) | [`artifacts/modelo_avaliado.joblib`](../artifacts/modelo_avaliado.joblib) | sim | hash confere |
| Modelo e avaliação (congelados) | [`artifacts/avaliacao_temporal.json`](../artifacts/avaliacao_temporal.json) | sim | — |
| Modelo e avaliação (congelados) | [`reports/metricas_modelagem.json`](../reports/metricas_modelagem.json) | sim | hash confere |
| Modelo e avaliação (congelados) | [`reports/relatorio_modelagem.md`](../reports/relatorio_modelagem.md) | sim | hash confere |
| Modelo e avaliação (congelados) | [`reports/curvas_modelagem.png`](../reports/curvas_modelagem.png) | sim | hash confere |
| Análises de negócio | [`reports/relatorio_analises_negocio.md`](../reports/relatorio_analises_negocio.md) | sim | hash confere |
| Análises de negócio | [`reports/metricas_analises_negocio.json`](../reports/metricas_analises_negocio.json) | sim | — |
| Análises de negócio | [`reports/verificacao_final.md`](../reports/verificacao_final.md) | sim | — |
| Análises de negócio | [`reports/figures/01_defasagem.png`](../reports/figures/01_defasagem.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/02_ida.png`](../reports/figures/02_ida.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/03_associacoes.png`](../reports/figures/03_associacoes.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/04_ips.png`](../reports/figures/04_ips.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/05_ipp.png`](../reports/figures/05_ipp.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/06_ipv.png`](../reports/figures/06_ipv.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/07_inde.png`](../reports/figures/07_inde.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/08_pedras.png`](../reports/figures/08_pedras.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/09_modelo.png`](../reports/figures/09_modelo.png) | sim | hash confere |
| Análises de negócio | [`reports/figures/10_insights.png`](../reports/figures/10_insights.png) | sim | hash confere |
| Código reutilizado neste notebook | [`src/modelagem.py`](../src/modelagem.py) | sim | — |
| Código reutilizado neste notebook | [`src/preparacao_coortes.py`](../src/preparacao_coortes.py) | sim | — |
| Código reutilizado neste notebook | [`src/analises_negocio.py`](../src/analises_negocio.py) | sim | — |
| Código reutilizado neste notebook | [`src/rastreabilidade.py`](../src/rastreabilidade.py) | sim | — |
| Código reutilizado neste notebook | [`src/dados_pede.py`](../src/dados_pede.py) | sim | — |
| Código reutilizado neste notebook | [`src/relatorio_modelagem.py`](../src/relatorio_modelagem.py) | sim | — |
| Código reutilizado neste notebook | [`src/relatorio_analises.py`](../src/relatorio_analises.py) | sim | — |

Documentos de referência técnica citados nos relatórios: [imputação em pipelines](https://scikit-learn.org/stable/modules/impute.html) e [métricas de classificação](https://scikit-learn.org/stable/modules/model_evaluation.html).

---
*Fim do notebook. Este documento não gera nem altera artefatos: todos os arquivos referenciados foram apenas
lidos. Estado do plano e próximas TASKs: `docs/TASKS.md`.*